# MB5 · Les maths de la diffusion — notebook d'exercices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookMaths/blob/main/chapitres/mb5_diffusion/03_notebook.ipynb)

La partie 0 vérifie tes exercices papier. Chaque exercice de code : un énoncé, une cellule à compléter (les `...` et les `raise NotImplementedError`), puis une cellule de vérification (`wbm.check`). Dans les parties A à D, donne les valeurs que rend le calcul, **sans les arrondir** : les décimales demandées sont celles de ta copie. « Tout exécuter » (*Run all*) va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳. Bloqué 15 minutes ? `04_indices.md`.

> Travaille dans **ta copie** (`mon_travail/mb5_diffusion/03_notebook.ipynb`, créée par `python tools/start_chapter.py MB5`) ; la version du dépôt est mise à jour par Claude. Sur Colab, le badge ouvre la version du dépôt, qui n'est pas enregistrée : crée puis ouvre ta copie comme l'explique `00_setup/COLAB.md` §2.

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | MB5.1 à MB5.9, MB5.14 | Les exercices papier (vérification) | ✏️ 🔗 | | |
| A | MB5.16 | Propager la moyenne et la variance d'une récurrence linéaire gaussienne à coefficients quelconques · `linear_gaussian_recursion` | 🔨 | ★ | 30 |
|  | MB5.17 | Le débruiteur optimal d'une mesure bruitée quand le signal suit un mélange de gaussiennes, des points compris · `posterior_mean_mixture` | 🔨 | ★★ | 45 |
|  | MB5.18 | Suivre un champ de vitesses qui dépend du temps : la méthode d'Euler et celle du point milieu, dans les deux sens du temps · `euler_ode`, `midpoint_ode` | 🔨 | ★★ | 40 |
| B | MB5.19 | `scipy.integrate.solve_ivp` : une croissance logistique suivie par tes deux méthodes, l'erreur au temps final selon le nombre de pas en échelle log-log, et les pentes 1 et 2 | 📦 | ★ | 20 |
|  | MB5.20 | Prédire ce que rend le débruiteur optimal de cinq relevés quand le bruit tend vers 0, pour une mesure proche d'un relevé puis à égale distance de deux, et mesurer | 🔮 | ★ | 15 |
|  | MB5.21 | Le chemin droit vers une loi gaussienne : Euler et point milieu sur le champ moyen exact, de 1 à 32 pas, la moyenne et l'écart-type obtenus contre ceux de la cible | 🔬 | ★★ | 25 |
|  | MB5.22 | Des lois guidées p_c^{1+w} p_u^{−w} sur une grille : une loi générale à deux bosses, une loi de classe gaussienne, et ce que w resserre et déplace | 🎨 | ★ | 20 |
| C | MB5.23 | Chasse à l'erreur : un pont aux poids inversés, un débruiteur qui oublie le bruit dans ses responsabilités, et un Euler dont le temps n'avance pas | 🐛 | ★ | 20 |
|  | MB5.24 | Un test de propriété : Euler exact pour une vitesse constante, le point milieu exact pour une vitesse affine en t, et l'erreur d'Euler divisée par deux avec le pas | 🛠️ | ★ | 30 |
| D | MB5.25 | Défi : transporter un bruit N(0, 1) vers une loi à deux bosses par son champ moyen exact, en au plus huit évaluations du champ, sous une distance de Wasserstein fixée | 🏆 | ★★★ | 35 |

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (a few seconds per cell); False = bigger experiments
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookMaths.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookMaths")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WBM_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wbm").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wbm").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier de workbookMaths.")

os.environ["WBM_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wbm  # noqa: E402

cfg = wbm.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="MB5")  # your own library

## Objectifs et rappel express

- Vérifier tes calculs papier (partie 0) : l'anomalie de température d'un lac, la survie d'une ruche, un phoque balisé, une voix et un bruit de fond, un signal reçu, un bit relayé, la floraison d'un verger, une tasse de thé, un chemin droit vers une loi gaussienne, et un pont vers une loss de débruitage.
- Écrire les quatre fonctions de `mymath/gaussian_paths.py` (`linear_gaussian_recursion`, `posterior_mean_mixture`, `euler_ode`, `midpoint_ode`) et les tester.
- Mesurer l'ordre de deux méthodes avec `solve_ivp`, prédire puis mesurer le débruiteur d'un nuage de points, transporter une loi par un champ moyen, tracer des lois guidées, corriger trois erreurs classiques, écrire trois tests de propriété, et relever un défi.

**Rappel express.** Pour $x_t = a_tx_{t-1} + b_t\varepsilon_t$ : $m_t = a_tm_{t-1}$ et $v_t = a_t^2v_{t-1} + b_t^2$ ; pour $a_t = a$, $\lvert a\rvert < 1$, $b_t = b$ : $v_\infty = \frac{b^2}{1 - a^2}$. Pour $y = ax + bn$ et $x$ de loi $\sum_k\pi_k\mathcal{N}(\mu_k, v_k)$, avec la variance $v_k$ de la fiche ($s_k^2$ dans la docstring, et dans les énoncés qui la suivent) : la composante $k$ a la responsabilité $\propto \pi_k\,\mathcal{N}(y \mid a\mu_k, a^2v_k + b^2)$, et $\mathbb{E}[x \mid y, k] = \mu_k + \frac{av_k}{a^2v_k + b^2}(y - a\mu_k)$. La loi guidée est proportionnelle à $p_c^{1+w}p_u^{-w}$. Euler : $x_{k+1} = x_k + h\,f(t_k, x_k)$, d'ordre 1 ; le point milieu : $x_{k+1} = x_k + h\,f\big(t_k + \frac h2, x_k + \frac h2f(t_k, x_k)\big)$, d'ordre 2, deux évaluations par pas ; $h = \frac{t_N - t_0}{N}$. Le chemin droit $x_s = (1 - s)\,x_0 + s\,\varepsilon$ a le champ moyen $v(x, s) = \frac{x - \mathbb{E}[x_0 \mid x_s = x]}{s}$, que l'on suit de $s = 1$ à $s = 0$.

## Partie 0 · Vérifier tes exercices papier (✏️ MB5.1 à MB5.9 ; 🔗 MB5.14)

Fais d'abord les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse ici : **la valeur** que tu as trouvée, sous la forme indiquée en commentaire, jamais le calcul qui y mène :
- un entier, ou un nombre avec un point décimal, `-1.25`, arrondi comme le demande la question ;
- une liste `[0.2, -1]`, dans l'ordre que dit la question.

Les réponses pas encore remplies affichent ⏳.

**Ex MB5.1 — L'anomalie de température d'un lac, un processus autorégressif : sa loi sachant aujourd'hui, sa loi stationnaire, le nombre de jours pour oublier le départ, et sa loi sachant un jour passé**

In [ ]:
# Ex MB5.1: the values you found on paper
answer_MB5_1a = ...  # a) a list of 2 numbers (4 decimals): the mean, then the variance of x_1
answer_MB5_1b = ...  # b) a list of 2 numbers (4 decimals): the mean, then the variance of x_3
answer_MB5_1c = ...  # c) a number (4 decimals)
answer_MB5_1d = ...  # d) an integer
answer_MB5_1e = ...  # e) a list of 2 numbers (4 decimals): the mean, then the variance of x_25 given x_20

for letter, answer in zip("abcde", [answer_MB5_1a, answer_MB5_1b, answer_MB5_1c, answer_MB5_1d, answer_MB5_1e]):
    wbm.check(f"MB5.1{letter}", answer)

**Ex MB5.2 — La survie d'une ruche à six mois de risques différents : le produit exact, son logarithme, le risque mensuel constant équivalent, et l'encadrement par deux exponentielles**

In [ ]:
# Ex MB5.2: the values you found on paper
answer_MB5_2a = ...  # a) a number (4 decimals)
answer_MB5_2b = ...  # b) a number (4 decimals)
answer_MB5_2c = ...  # c) a number (4 decimals)
answer_MB5_2d = ...  # d) a number (4 decimals)
answer_MB5_2e = ...  # e) a number (4 decimals)

for letter, answer in zip("abcde", [answer_MB5_2a, answer_MB5_2b, answer_MB5_2c, answer_MB5_2d, answer_MB5_2e]):
    wbm.check(f"MB5.2{letter}", answer)

**Ex MB5.3 — Un phoque balisé entre deux relevés : sa loi à une heure intermédiaire (le pont d'une marche aléatoire), puis la corrélation de deux heures pour des scénarios aux mêmes lois**

In [ ]:
# Ex MB5.3: the values you found on paper
answer_MB5_3a = ...  # a) a list of 2 numbers (4 decimals): the mean, then the variance at 11 h
answer_MB5_3b = ...  # b) a list of 2 numbers (3 decimals): the hour (an integer), then the variance
answer_MB5_3c = ...  # c) a number (4 decimals)
answer_MB5_3d = ...  # d) a number (4 decimals)

for letter, answer in zip("abcd", [answer_MB5_3a, answer_MB5_3b, answer_MB5_3c, answer_MB5_3d]):
    wbm.check(f"MB5.3{letter}", answer)

**Ex MB5.4 — Une voix mêlée à un bruit de fond : SNR, part de variance de la voix, logit et décibels, puis la voix retrouvée à partir d'une estimation du bruit, et l'erreur amplifiée**

In [ ]:
# Ex MB5.4: the values you found on paper
answer_MB5_4a = ...  # a) a number (2 decimals)
answer_MB5_4b = ...  # b) a number (4 decimals)
answer_MB5_4c = ...  # c) a number (4 decimals)
answer_MB5_4d = ...  # d) a number (1 decimal)
answer_MB5_4e = ...  # e) a number (4 decimals)
answer_MB5_4f = ...  # f) a number (4 decimals)
answer_MB5_4g = ...  # g) a list of 2 numbers (4 decimals): recording 1, then recording 2

for letter, answer in zip("abcdefg", [answer_MB5_4a, answer_MB5_4b, answer_MB5_4c, answer_MB5_4d, answer_MB5_4e, answer_MB5_4f, answer_MB5_4g]):
    wbm.check(f"MB5.4{letter}", answer)

**Ex MB5.5 — Débruiter un signal reçu, d'abord gaussien, puis émis à deux niveaux ±c : E[x | y] par M25a et par la formule de Tweedie, et la limite quand le bruit tend vers 0**

In [ ]:
# Ex MB5.5: the values you found on paper
answer_MB5_5a = ...  # a) a number (4 decimals)
answer_MB5_5b = ...  # b) a number (4 decimals)
answer_MB5_5c = ...  # c) a number (4 decimals)
answer_MB5_5d = ...  # d) a number (4 decimals)
answer_MB5_5e = ...  # e) a number (4 decimals)
answer_MB5_5f = ...  # f) a number (4 decimals)

for letter, answer in zip("abcdef", [answer_MB5_5a, answer_MB5_5b, answer_MB5_5c, answer_MB5_5d, answer_MB5_5e, answer_MB5_5f]):
    wbm.check(f"MB5.5{letter}", answer)

**Ex MB5.6 — Un bit altéré par deux relais, puis reconstitué par un modèle retour : ln p(x), l'ELBO de la loi aller et sa décomposition en reconstruction, KL finale et KL d'un pas retour**

In [ ]:
# Ex MB5.6: the values you found on paper
answer_MB5_6a = ...  # a) a list of 2 numbers (4 decimals): q(z2 = 0 | x = 1), then q(z2 = 1 | x = 1)
answer_MB5_6b = ...  # b) a list of 2 numbers (4 decimals): q(z1 = 1 | z2 = 0, x = 1), then q(z1 = 1 | z2 = 1, x = 1)
answer_MB5_6c = ...  # c) a number (4 decimals)
answer_MB5_6d = ...  # d) a number (4 decimals): by its decomposition, the reconstruction minus the two KL of e)
answer_MB5_6e = ...  # e) a list of 3 numbers (4 decimals): the reconstruction, the final KL, the KL of the backward step

for letter, answer in zip("abcde", [answer_MB5_6a, answer_MB5_6b, answer_MB5_6c, answer_MB5_6d, answer_MB5_6e]):
    wbm.check(f"MB5.6{letter}", answer)

**Ex MB5.7 — La date de floraison d'une variété et celle de tout le verger : la loi p_v^{1+w} p_g^{−w} pour deux valeurs de w, son score, sa condition d'existence et le w limite**

In [ ]:
# Ex MB5.7: the values you found on paper
answer_MB5_7a = ...  # a) a number (4 decimals)
answer_MB5_7b = ...  # b) a number (4 decimals)
answer_MB5_7c = ...  # c) a list of 2 numbers (4 decimals): the mean, then the variance
answer_MB5_7d = ...  # d) a number (4 decimals)
answer_MB5_7e = ...  # e) a number (4 decimals)

for letter, answer in zip("abcde", [answer_MB5_7a, answer_MB5_7b, answer_MB5_7c, answer_MB5_7d, answer_MB5_7e]):
    wbm.check(f"MB5.7{letter}", answer)

**Ex MB5.8 — Une tasse de thé qui refroidit : Euler et point milieu contre la solution exacte, l'erreur quand le pas est divisé par deux, et un calcul à rebours**

In [ ]:
# Ex MB5.8: the values you found on paper
answer_MB5_8a = ...  # a) a number (2 decimals)
answer_MB5_8b = ...  # b) a list of 2 numbers (2 decimals): h = 5, then h = 2.5
answer_MB5_8c = ...  # c) a list of 2 numbers (2 decimals): h = 5, then h = 2.5
answer_MB5_8d = ...  # d) a list of 2 numbers (1 decimal): Euler, then the midpoint method, from the unrounded values of a), b), c)
answer_MB5_8e = ...  # e) a list of 2 numbers (2 decimals): one Euler step backward, then the exact value

for letter, answer in zip("abcde", [answer_MB5_8a, answer_MB5_8b, answer_MB5_8c, answer_MB5_8d, answer_MB5_8e]):
    wbm.check(f"MB5.8{letter}", answer)

**Ex MB5.9 — Le chemin droit d'un bruit N(0, 1) vers une loi N(m, τ²) : le champ moyen, un pas d'Euler depuis s = 1, puis deux pas, et l'écart-type obtenu contre τ**

In [ ]:
# Ex MB5.9: the values you found on paper
answer_MB5_9a = ...  # a) a number (4 decimals)
answer_MB5_9b = ...  # b) a number (4 decimals)
answer_MB5_9c = ...  # c) a number (4 decimals)
answer_MB5_9d = ...  # d) a number (4 decimals)
answer_MB5_9e = ...  # e) a list of 2 numbers (4 decimals): the standard deviation, then the exact point reached

for letter, answer in zip("abcde", [answer_MB5_9a, answer_MB5_9b, answer_MB5_9c, answer_MB5_9d, answer_MB5_9e]):
    wbm.check(f"MB5.9{letter}", answer)

**Ex MB5.14 — Pont : une loss de débruitage à trois niveaux de bruit, son minimiseur à chaque niveau, et les MSE minimales sur le signal et sur le bruit**

In [ ]:
# Ex MB5.14: the values you found on paper
answer_MB5_14a = ...  # a) a number (4 decimals)
answer_MB5_14b = ...  # b) a list of 3 numbers (4 decimals): levels 1, 2, 3
answer_MB5_14c = ...  # c) a list of 3 numbers (4 decimals): levels 1, 2, 3
answer_MB5_14d = ...  # d) a number (4 decimals)

for letter, answer in zip("abcd", [answer_MB5_14a, answer_MB5_14b, answer_MB5_14c, answer_MB5_14d]):
    wbm.check(f"MB5.14{letter}", answer)

## Partie A · Ta librairie : une récurrence linéaire gaussienne, le débruiteur d'un mélange, Euler et le point milieu

Fiche §1, §5 et §8. Les quatre fonctions de `mymath/gaussian_paths.py`, leurs tests, et un usage de chacune. La cellule de données ci-dessous recopie les données des exercices papier.

In [ ]:
# Tools of the notebook (parts A to D)
import contextlib
import dis
import math
import os
import re
import sys
import time
from types import MappingProxyType

import matplotlib.pyplot as plt
import numpy as np

from wbm.errors import MymathMissingError
from wbm.nbtools import StopCheck, checked, drawn, filled, fr, run_pytest_group, verdict

TEST_MB5 = "tests/test_mb5_gaussian_paths.py"          # the tests of mymath/gaussian_paths.py
REF = wbm.load_mymath("ref", alias="mymath_ref")       # the reference: the checks of the hidden cases rely on it
ref_gp = REF.gaussian_paths


def shown(value, digits=4):
    """A short text for a result of the learner's code (the check messages): numbers rounded, as lists."""
    if value is None:
        return "None"
    try:
        return str(np.round(np.asarray(value, dtype=float), digits).tolist())[:160]
    except (TypeError, ValueError):
        return repr(value)[:160]


def check_answer(ex_id, value, hint=None):
    """wbm.check(ex_id, value, computed=True). For a list that holds True or False, the message gives only how many
    elements are right, not the place of the first wrong one (it would let you guess the booleans one by one).
    hint(value), when given, may return a targeted « Piste » for a wrong answer (None otherwise)."""
    holds_bool = isinstance(value, (list, tuple)) and any(isinstance(v, (bool, np.bool_)) or (
        isinstance(v, (list, tuple)) and any(isinstance(w, (bool, np.bool_)) for w in v)) for v in value)
    if not holds_bool and hint is None:
        return wbm.check(ex_id, value, computed=True)
    result = wbm.check(ex_id, value, computed=True, quiet=True)
    message = result.message
    if holds_bool:
        for cut in (" Le premier faux est le ", " Premier élément faux à l'indice "):
            message = message.split(cut)[0]
    if hint is not None and result.status not in ("correct", "pending", "unknown") and "Piste" not in message:
        try:
            extra = hint(value)
        except Exception:  # noqa: BLE001 - an odd answer: no targeted hint
            extra = None
        if extra:
            message = f"{message} Piste : {extra}"
    icon = {"correct": "✅", "pending": "⏳", "unknown": "❓"}.get(result.status, "❌")
    print(f"{icon} Ex {ex_id} : {message}")
    return result


def library_hint(names):
    """A hint for a wrong answer computed with the learner's library (no classic error recognised): its tests first,
    then the call."""
    return lambda value: (f"cette valeur vient de {names} de ta librairie : si ses tests passent tous (plus haut), "
                          "l'erreur est dans l'appel (les arguments, leur ordre, leurs unités ; quel pas tu lis) ; et "
                          "donne la valeur que rend le calcul, sans l'arrondir")


def passes(ex_id, value):
    """True when value is the hidden answer ex_id (wbm.check, computed=True, nothing printed)."""
    try:
        return bool(wbm.check(ex_id, value, computed=True, quiet=True).passed)
    except Exception:  # noqa: BLE001 - an odd value is not the answer
        return False


def reporter(results, stop=True):
    """A function report(ok, success, failure) that prints one ✅ or ❌ line, keeps the verdict in results, and (stop)
    ends the check at the first ❌ (StopCheck, caught by checked)."""
    def report(ok, success, failure):
        results.append(bool(ok))
        print(("✅ " + success) if ok else ("❌ " + failure))
        if not ok and stop:
            raise StopCheck
    return report


def as_floats(value):
    """value as a float array, or None when it is not an array of numbers."""
    try:
        return np.asarray(value, dtype=float)
    except (TypeError, ValueError):
        return None


def matches(got, want, rtol=1e-9, atol=1e-12):
    """True when got is an array of numbers of the shape of want and equal to it up to rounding (no error for an odd
    got)."""
    arr = as_floats(got)
    want = np.asarray(want, dtype=float)
    return arr is not None and arr.shape == want.shape and bool(np.allclose(arr, want, rtol=rtol, atol=atol,
                                                                            equal_nan=True))


def line_with(ax, xs, ys, rtol=1e-7, atol=1e-9):
    """The first line of ax that has exactly the abscissas xs and the ordinates ys, or None."""
    xs, ys = np.asarray(xs, dtype=float), np.asarray(ys, dtype=float)
    for line in ax.lines:
        x, y = np.asarray(line.get_xdata(), dtype=float), np.asarray(line.get_ydata(), dtype=float)
        if x.shape == xs.shape and y.shape == ys.shape and np.allclose(x, xs, rtol=rtol, atol=atol) \
                and np.allclose(y, ys, rtol=rtol, atol=atol, equal_nan=True):
            return line
    return None


def call_safely(label, function, *args, **kwargs):
    """function(*args, **kwargs); an error of the learner's code becomes one ❌ line naming the call (StopCheck)."""
    try:
        return function(*args, **kwargs)
    except (NotImplementedError, MymathMissingError):
        raise
    except Exception as exc:  # noqa: BLE001 - the learner's error, reported with the name of the call
        print(f"❌ {label} : ton code lève {type(exc).__name__} : {str(exc)[:200]}")
        raise StopCheck from None


def frozen(array):
    """A float copy of array that refuses any change in place (the data given to the learner's functions)."""
    out = np.array(array, dtype=float)
    out.setflags(write=False)
    return out


class CodeFrom(BaseException):
    """Raised inside code_from when watched code starts (a BaseException: a broad except of the learner's code does
    not hide it)."""


_NOT_KEYED = {"CACHE", "PUSH_NULL", "RESUME", "NOP", "EXTENDED_ARG", "NOT_TAKEN"}


def code_key(code):
    """What identifies a compiled function whatever its file and the module around it: its instructions (without jump
    targets, nor what depends on the rest of the module) and the names of its variables."""
    ops = []
    for i in dis.get_instructions(code):
        if i.opname in _NOT_KEYED:
            continue
        jump = i.opcode in dis.hasjrel or i.opcode in dis.hasjabs or "JUMP" in i.opname or i.opname == "FOR_ITER"
        ops.append((i.opname, None if jump else i.argval.co_name if hasattr(i.argval, "co_code") else repr(i.argval)))
    return tuple(ops), code.co_varnames


def code_keys_of_file(path):
    """The code_key of every function of a Python file, compiled but not run (to recognise a copy of its code)."""
    keys, todo = set(), [compile(open(path, encoding="utf-8").read(), str(path), "exec")]
    while todo:
        code = todo.pop()
        todo += [c for c in code.co_consts if hasattr(c, "co_code")]
        key = code_key(code)
        if code.co_name != "<module>" and len(key[0]) >= 15:
            keys.add(key)
    return keys


_ESCAPE = {"watch": None}      # what code_from(..., escapes=True) watches now (None: nothing)
_PROCESS_EVENTS = ("subprocess.Popen", "os.system", "os.posix_spawn", "os.fork", "os.forkpty", "os.exec", "os.spawn")


def _escape_hook(event, args, escape=_ESCAPE, processes=_PROCESS_EVENTS, os=os, stop=CodeFrom):
    """Audit hook, for the life of the kernel: inside code_from(..., escapes=True), opening a file of the watched places
    or starting another process stops the block (code run by another process escapes sys.monitoring). Everything it
    uses is bound when it is defined: it also runs while the kernel shuts down."""
    watch = escape["watch"]
    if watch is None:
        return
    if event == "open" and args and isinstance(args[0], (str, bytes, os.PathLike)):
        real = os.path.realpath(os.fsdecode(args[0]))
        if any(real == place or real.startswith(place + os.sep) for place in watch["places"]):
            watch["seen"].append("ouvre " + os.path.basename(real))
            raise stop
    elif event.startswith(processes):
        watch["seen"].append("lance un autre processus : " + event)
        raise stop


sys.addaudithook(_escape_hook)


@contextlib.contextmanager
def code_from(places, twins=(), escapes=False):
    """Inside the block, the start of any Python code (a function or a module body) of a file among places (files, or
    folders with everything under them), or of a function whose code_key is among twins (a copy of such a function),
    stops the block; yields the list of the names seen ("gaussian_paths.euler_ode"). The learner's function must not run
    that code (mymath in a corrected function, the hidden module of a challenge). With escapes, opening a file of places
    or starting another process also stops it ("ouvre mb5_25.py", "lance un autre processus : subprocess.Popen").
    Without sys.monitoring (Python < 3.12) or a free tool id, code is not watched."""
    places = [os.path.realpath(str(place)) for place in places]
    libraries = tuple({os.path.realpath(sys.prefix) + os.sep, os.path.realpath(sys.base_prefix) + os.sep})
    monitoring, tool, seen, inside = getattr(sys, "monitoring", None), None, [], {}
    for candidate in ((4, 3, 2) if monitoring is not None else ()):
        try:
            monitoring.use_tool_id(candidate, "code_from")
        except ValueError:                               # taken by another tool
            continue
        tool = candidate
        break
    if tool is not None:
        def on_start(code, offset):
            name = code.co_filename
            if name not in inside:
                real = os.path.realpath(name) if os.path.exists(name) else name
                if any(real == place or real.startswith(place + os.sep) for place in places):
                    inside[name] = True                  # watched
                elif twins and not real.startswith(libraries):
                    inside[name] = False                 # the learner's code: compared with twins
                else:
                    inside[name] = None                  # an installed library: not compared
            if inside[name]:
                seen.append(f"{os.path.splitext(os.path.basename(name))[0]}.{code.co_name}")
                raise CodeFrom
            if inside[name] is False and code_key(code) in twins:
                seen.append(f"une copie de {code.co_name}")
                raise CodeFrom
            return monitoring.DISABLE                    # other code: not watched again until the end of the block

        monitoring.register_callback(tool, monitoring.events.PY_START, on_start)
        monitoring.set_events(tool, monitoring.events.PY_START)
    if escapes:
        _ESCAPE["watch"] = {"places": places, "seen": seen}
    try:
        yield seen
    except CodeFrom:
        pass
    finally:
        _ESCAPE["watch"] = None
        if tool is not None:
            monitoring.set_events(tool, 0)
            monitoring.register_callback(tool, monitoring.events.PY_START, None)
            monitoring.free_tool_id(tool)
            monitoring.restart_events()


# The data of the paper exercises (02_exercices.md), read only
LAKE_1 = MappingProxyType({"a": 0.88, "b": 0.35, "x0": 2.5})       # MB5.1: x_t = a x_{t-1} + b eps_t, x_0 known (°C)
SEAL_3 = MappingProxyType({"x8h": 3.0, "x16h": 9.0, "var_8h_11h": 2.0, "var_11h_16h": 0.5})   # MB5.3: positions (km)
#   at 8 h and 16 h, and the variance (km²) that each hour adds, from 8 h to 11 h, then from 11 h to 16 h
SIGNAL_5A = MappingProxyType({"m": 1.0, "v": 4.0, "a": 2.0, "b": 3.0, "y": 6.0})   # MB5.5 A: x ~ N(m, v), y = a x + b n
SIGNAL_5B = MappingProxyType({"c": 2.0, "a": 0.5, "b": 1.0, "y": 1.1})   # MB5.5 B: x = +c or -c (1/2 each), y = a x + b n
TEA_8 = MappingProxyType({"r": 0.06, "room": 20.0, "T0": 85.0, "T10": 47.0})   # MB5.8: T' = -r (T - room), T(0) = T0
#   (°C, minutes); e) another cup at T10 at 10 min
DATA_9 = MappingProxyType({"m": 4.0, "tau": 0.5})                      # MB5.9: the data x_0 ~ N(m, tau²)

### Ex MB5.16 — Propager la moyenne et la variance d'une récurrence linéaire gaussienne à coefficients quelconques · `linear_gaussian_recursion` 🔨 ★ ⏱️ 30 min
**Objectif :** programmer la propagation de la moyenne et de la variance d'une récurrence linéaire gaussienne, puis retrouver MB5.1 et suivre des coefficients qui changent à chaque pas, ou négatifs.  
**Prérequis :** fiche §1 ; M25a (image affine d'une gaussienne), R5 (suite géométrique) ; Ex MB5.1 · **mymath :** `gaussian_paths.py` · **Parcours :** C, E

> **Mode d'emploi `mymath`** (comme en MB4). Ouvre ta copie `mon_travail/mymath/gaussian_paths.py`, lis la docstring du module (ses conventions, et ce qui est permis) puis celle de la fonction, remplace chaque `raise NotImplementedError(...)` par ton code, **enregistre** le fichier, puis exécute les cellules ci-dessous : la première recharge ta librairie. Dans le notebook, appelle tes fonctions par `mymath.gaussian_paths.<nom>` (par exemple `mymath.gaussian_paths.euler_ode(f, 1.0, 0.0, 2.0, 4)`), ou donne-leur un nom court (`euler_ode = mymath.gaussian_paths.euler_ode`) dans une cellule exécutée après le rechargement. ⚠️ Le squelette n'importe rien : ajoute en tête du fichier `import math` et `import numpy as np`, et `from .approx import logsumexp` si tu reprends ton log-sum-exp de M05a. NumPy et `math` sont permis, mais aucune SciPy (`solve_ivp`, `odeint`, `quad`, `norm`, `logsumexp`, `softmax`) : les tests comparent tes résultats à SciPy et à des simulations, et un test lit ton module. Ne modifie jamais les tableaux reçus. Hors de `mymath`, le notebook peut se servir de tout NumPy et de SciPy.

Écris `linear_gaussian_recursion(a, b, mean0=0.0, var0=0.0)` (fiche §1), en suivant sa docstring : contrôle les arguments (deux tableaux 1-D de même longueur, de nombres finis ; `mean0` et `var0` réels finis, ni booléens, `var0` $\ge 0$), puis propage $m_t = a_tm_{t-1}$ et $v_t = a_t^2v_{t-1} + b_t^2$, et rends les deux tableaux de longueur $T + 1$ (l'entrée 0 est celle de $x_0$). Lance le groupe de tests.

**L'usage.** Avec ta fonction :
- a) le lac de MB5.1 (`LAKE_1`, cellule de données) sur trois jours : la moyenne et la variance de $x_3$ (une liste de 2 nombres, à comparer à ta copie) ;
- b) au printemps, la persistance et le bruit changent chaque semaine : le notebook fournit `A_16` et `B_16`, les coefficients de douze semaines, et le départ suit $x_0 \sim \mathcal{N}(1{,}5 ;\ 0{,}3)$ : la moyenne et la variance de $x_{12}$ (une liste de 2 nombres) ;
- c) $a = -0{,}68$ et $b = 1{,}1$ pendant sept pas, depuis $x_0 = 5$ connu : la moyenne et la variance de $x_7$, puis $v_\infty$ (une liste de 3 nombres).

In [ ]:
# Spring: the coefficients of twelve weeks (the persistence a_t and the noise b_t change every week)
A_16 = frozen([0.96, 0.93, 0.88, 0.71, 0.58, 1.04, 0.99, 0.83, 0.66, 0.91, 1.02, 0.79])
B_16 = frozen([0.36, 0.21, 0.57, 0.52, 0.38, 0.26, 0.34, 0.41, 0.55, 0.48, 0.28, 0.44])

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="MB5")  # reload your saved file

**Les tests** de `linear_gaussian_recursion` (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group(TEST_MB5, "test_linear_gaussian_recursion_ or test_module_uses_")

**L'usage** : le lac de MB5.1, douze semaines de printemps, puis un coefficient négatif.

In [ ]:
answer_MB5_16a = answer_MB5_16b = answer_MB5_16c = ...
with wbm.attempt("MB5.16"):          # if you call linear_gaussian_recursion here: ⏳ while it is not written
    answer_MB5_16a = ...  # a) a list of 2 numbers, as computed: the mean and the variance of x_3 (the lake of MB5.1)
    answer_MB5_16b = ...  # b) a list of 2 numbers, as computed: the mean and the variance of x_12 (A_16, B_16, the start)
    answer_MB5_16c = ...  # c) a list of 3 numbers, as computed: the mean and the variance of x_7, then v_inf

In [ ]:
check_answer("MB5.16a", answer_MB5_16a, hint=library_hint("linear_gaussian_recursion"))
check_answer("MB5.16b", answer_MB5_16b, hint=library_hint("linear_gaussian_recursion"))
check_answer("MB5.16c", answer_MB5_16c, hint=library_hint("linear_gaussian_recursion"))

**Ce qu'il faut regarder** : pourquoi les moyennes de c) changent-elles de signe à chaque pas, et de combien leur valeur absolue diminue-t-elle ? Par quel côté la variance approche-t-elle $v_\infty$ ? Les réponses sont dans `05_solutions.md`, section Notebook.

### Ex MB5.17 — Le débruiteur optimal d'une mesure bruitée quand le signal suit un mélange de gaussiennes, des points compris · `posterior_mean_mixture` 🔨 ★★ ⏱️ 45 min
**Objectif :** programmer le débruiteur optimal d'une mesure bruitée quand le signal suit un mélange de gaussiennes (des points compris), puis retrouver MB5.5 et débruiter un signal à deux bosses.  
**Prérequis :** fiche §5 ; M25a §5 (modèle linéaire gaussien), M25b (responsabilités), M05a (log-sum-exp) ; Ex MB5.5 · **mymath :** `gaussian_paths.py` · **Parcours :** C, E

> Même mode d'emploi qu'en MB5.16 : écris les fonctions dans ta copie `mon_travail/mymath/gaussian_paths.py`, **enregistre**, puis exécute les cellules ci-dessous ; la première recharge ta librairie.

Écris `posterior_mean_mixture(y, a, b, weights, means, variances)` (fiche §5), en suivant sa docstring : contrôle les arguments ; pour chaque composante de variance $v_k$ ($s_k^2$ dans la docstring), la variance $a^2v_k + b^2$ de $y$, le logarithme de $\pi_k\,\mathcal{N}(y \mid a\mu_k, a^2v_k + b^2)$ ($-\infty$ pour un poids nul) et $\mathbb{E}[x \mid y, k]$ ; les responsabilités par un log-sum-exp sur $k$, pour chaque mesure (le `logsumexp` de ton `approx.py`, M05a, ou, à défaut, un `_logsumexp` privé écrit dans `gaussian_paths.py`) ; enfin la moyenne pondérée, de la forme de `y` ; pour $a = 0$, la moyenne du mélange. Lance le groupe de tests.

**L'usage.** Avec ta fonction :
- a) la partie A de MB5.5 (`SIGNAL_5A`, cellule de données), comme un mélange à une composante : $\mathbb{E}[x \mid y = 6]$ (un nombre, à comparer à ta copie) ;
- b) la partie B de MB5.5 (`SIGNAL_5B`), comme deux points de poids $\frac12$ (des variances nulles) : $\mathbb{E}[x \mid y = 1{,}1]$ (un nombre) ;
- c) un signal reçu qui suit le mélange $0{,}25\,\mathcal{N}(-1{,}4 ;\ 0{,}2) + 0{,}75\,\mathcal{N}(2{,}3 ;\ 0{,}5)$, mesuré par $y = 0{,}7\,x + 0{,}5\,n$ : $\mathbb{E}[x \mid y]$ pour $y = -0{,}9$, $0{,}3$ et $1{,}7$ (une liste de 3 nombres).

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="MB5")  # reload your saved file

**Les tests** de `posterior_mean_mixture` (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group(TEST_MB5, "test_posterior_mean_mixture_")

**L'usage** : les deux signaux de MB5.5, puis un signal à deux bosses.

In [ ]:
answer_MB5_17a = answer_MB5_17b = answer_MB5_17c = ...
with wbm.attempt("MB5.17"):          # if you call posterior_mean_mixture here: ⏳ while it is not written
    answer_MB5_17a = ...  # a) a number, as computed: E[x | y = 6] for the part A of MB5.5 (a mixture of one component)
    answer_MB5_17b = ...  # b) a number, as computed: E[x | y = 1.1] for the part B of MB5.5 (two points, weights 1/2)
    answer_MB5_17c = ...  # c) a list of 3 numbers, as computed: E[x | y] for y = -0.9, 0.3, 1.7 (the mixture of c)

In [ ]:
check_answer("MB5.17a", answer_MB5_17a, hint=library_hint("posterior_mean_mixture"))
check_answer("MB5.17b", answer_MB5_17b, hint=library_hint("posterior_mean_mixture"))
check_answer("MB5.17c", answer_MB5_17c, hint=library_hint("posterior_mean_mixture"))

**Ce qu'il faut regarder** : pour $y = -0{,}9$, quelle composante l'emporte, malgré son petit poids, et pourquoi $\mathbb{E}[x \mid y]$ n'est-il pas $\frac ya$ ? Que rend ta fonction pour $a = 0$ ? Les réponses sont dans `05_solutions.md`, section Notebook.

### Ex MB5.18 — Suivre un champ de vitesses qui dépend du temps : la méthode d'Euler et celle du point milieu, dans les deux sens du temps · `euler_ode`, `midpoint_ode` 🔨 ★★ ⏱️ 40 min
**Objectif :** programmer la méthode d'Euler et celle du point milieu pour un champ qui dépend du temps, dans les deux sens du temps, puis retrouver MB5.8 et voir quand chacune est exacte.  
**Prérequis :** fiche §8 ; M20 §6 (Euler), M02a (sommes de Riemann, point milieu) ; Ex MB5.8 · **mymath :** `gaussian_paths.py` · **Parcours :** C, E

> Même mode d'emploi qu'en MB5.16 : écris les fonctions dans ta copie `mon_travail/mymath/gaussian_paths.py`, **enregistre**, puis exécute les cellules ci-dessous ; la première recharge ta librairie.

Écris `euler_ode(f, x0, t0, t1, n_steps)` et `midpoint_ode(f, x0, t0, t1, n_steps)` (fiche §8), en suivant leurs docstrings : contrôle `n_steps` (un entier $\ge 1$, pas un booléen ni un flottant), `t0` et `t1` (des réels finis) et la forme de chaque vitesse ; $h = \frac{t_N - t_0}{N}$, négatif si $t_N < t_0$ (l'argument `t1`, comme celui de `solve_ivp`, est l'instant final $t_N$) ; une évaluation de `f` par pas pour Euler, deux pour le point milieu ; rends les $N + 1$ points, départ compris. Lance les deux groupes de tests.

**L'usage.** Avec tes fonctions :
- a) le thé de MB5.8 (`TEA_8`, cellule de données), à 10 min, par Euler puis par le point milieu, avec deux pas (une liste de 2 nombres, à comparer à ta copie) ;
- b) la tasse à 47 °C de MB5.8 e), remontée de 10 min à 5 min en dix pas d'Euler : sa température à 5 min (un nombre) ;
- c) $x' = 3t^2$, $x(0) = 0$, sur $[0, 2]$ en quatre pas : $x(2)$ par Euler, puis par le point milieu (une liste de 2 nombres) ;
- d) $x' = t - x$, $x(0) = 2$, sur $[0, 3]$ en six pas : l'erreur d'Euler, puis celle du point milieu, en $t = 3$, par rapport à la solution exacte $t - 1 + 3e^{-t}$ (une liste de 2 nombres, positifs).

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="MB5")  # reload your saved file

**Les tests** de `euler_ode`, puis ceux de `midpoint_ode` (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group(TEST_MB5, "test_euler_ode_")

In [ ]:
run_pytest_group(TEST_MB5, "test_midpoint_ode_")

**L'usage** : le thé de MB5.8, dans un sens puis dans l'autre, puis deux champs dont on connaît la solution.

In [ ]:
answer_MB5_18a = answer_MB5_18b = answer_MB5_18c = answer_MB5_18d = ...
with wbm.attempt("MB5.18"):          # if you call your functions here: ⏳ while they are not written
    answer_MB5_18a = ...  # a) a list of 2 numbers, as computed: the tea at 10 min in two steps, Euler then midpoint
    answer_MB5_18b = ...  # b) a number, as computed: the cup at 47 °C at 10 min, taken back to 5 min in ten Euler steps
    answer_MB5_18c = ...  # c) a list of 2 numbers, as computed: x(2) for x' = 3t², four steps, Euler then midpoint
    answer_MB5_18d = ...  # d) a list of 2 numbers, as computed: the errors at t = 3 (positive), Euler then midpoint

In [ ]:
check_answer("MB5.18a", answer_MB5_18a, hint=library_hint("euler_ode et midpoint_ode"))
check_answer("MB5.18b", answer_MB5_18b, hint=library_hint("euler_ode et midpoint_ode"))
check_answer("MB5.18c", answer_MB5_18c, hint=library_hint("euler_ode et midpoint_ode"))
check_answer("MB5.18d", answer_MB5_18d, hint=library_hint("euler_ode et midpoint_ode"))

**Ce qu'il faut regarder** : à quelle somme de M02a correspondent les deux valeurs de c), et pourquoi le point milieu n'y est-il pas exact ? De combien l'erreur de b) en dix pas est-elle plus petite que celle d'un seul pas, en MB5.8 e) ? Les réponses sont dans `05_solutions.md`, section Notebook.

## Partie B · Une bibliothèque, prédire, mesurer, dessiner

Fiche §5, §7, §8 et §9. Une croissance logistique suivie par `solve_ivp` et par tes deux méthodes ; le débruiteur d'un nuage de points quand le bruit s'efface ; le chemin droit vers une loi gaussienne ; des lois guidées sur une grille.

### Ex MB5.19 — `scipy.integrate.solve_ivp` : une croissance logistique suivie par tes deux méthodes, l'erreur au temps final selon le nombre de pas en échelle log-log, et les pentes 1 et 2 📦 ★ ⏱️ 20 min
**Objectif :** suivre une croissance logistique par `solve_ivp` et par tes deux méthodes, et mesurer l'ordre de leur erreur en échelles logarithmiques.  
**Prérequis :** fiche §8 ; M02a §5 (ordre d'une méthode) ; Ex MB5.18 · **Parcours :** C

Une culture de levures suit $P' = 0{,}9\,P\big(1 - \frac{P}{40}\big)$ (en millions de cellules, $t$ en heures), depuis $P(0) = 1{,}5$ ; la fiche §8 donne la solution exacte de la croissance logistique. Avec `scipy.integrate.solve_ivp` et tes fonctions `euler_ode` et `midpoint_ode`, relève :
- a) $P(8)$ par `solve_ivp(…, rtol=1e-10, atol=1e-10)` (un nombre), à comparer à la solution exacte ;
- b) pour $N = 8$, 16, 32, 64, 128 et 256 pas, l'erreur de `euler_ode`, puis celle de `midpoint_ode`, en $t = 8$, par rapport à la solution exacte (deux listes de 6 nombres positifs : les valeurs absolues des écarts), et trace-les contre $N$ en échelles logarithmiques (`ax.loglog`) ;
- c) les pentes des deux droites, ajustées sur $N = 32$ à 256 par `np.polyfit` sur les logarithmes (une liste de 2 nombres, Euler puis le point milieu), puis lis dans b), à 64 évaluations du champ, l'erreur d'Euler (64 pas) et celle du point milieu (32 pas).

In [ ]:
answer_MB5_19a = answer_MB5_19b = answer_MB5_19c = ...
with wbm.attempt("MB5.19"):          # solve_ivp, then your euler_ode and midpoint_ode: ⏳ while they are not written
    answer_MB5_19a = ...  # a) a number, as computed: P(8) by solve_ivp(..., rtol=1e-10, atol=1e-10)
    answer_MB5_19b = ...  # b) two lists of 6 numbers, as computed: the errors at t = 8 of euler_ode, then of midpoint_ode, for N = 8 to 256
    answer_MB5_19c = ...  # c) a list of 2 numbers, as computed: the slopes fitted on N = 32 to 256, Euler then midpoint
    # and your figure: the errors of b) against N, in logarithmic scales

In [ ]:
check_answer("MB5.19a", answer_MB5_19a)
check_answer("MB5.19b", answer_MB5_19b, hint=library_hint("euler_ode et midpoint_ode"))
check_answer("MB5.19c", answer_MB5_19c, hint=library_hint("euler_ode et midpoint_ode"))

**Ce qu'il faut regarder** : pourquoi ajuster les pentes sur les grands $N$ seulement ? À nombre d'évaluations égal, quelle méthode gagne, et de combien ? Les réponses sont dans `05_solutions.md`, section Notebook.

### Ex MB5.20 — Prédire ce que rend le débruiteur optimal de cinq relevés quand le bruit tend vers 0, pour une mesure proche d'un relevé puis à égale distance de deux, et mesurer 🔮 ★ ⏱️ 15 min
**Objectif :** prévoir ce que rend le débruiteur optimal d'un nuage de points quand le bruit s'efface, puis le mesurer.  
**Prérequis :** fiche §5 ; M04 (Bayes), MB3 (moyenne pondérée par une softmax) ; Ex MB5.5, MB5.17 · **Parcours :** C, E

Un marégraphe a relevé cinq fois la hauteur de la mer, en m : $1{,}2$ ; $1{,}9$ ; $2{,}3$ ; $3{,}6$ ; $4{,}0$ (`RECORDS_20`), de poids égaux ; un capteur mesure $y = 0{,}5\,x + b\,n$, où $x$ est l'un des cinq relevés. **Avant d'exécuter quoi que ce soit**, prédis, quand $b$ tend vers 0 :
- a) pour $y = 1$ (donc $\frac ya = 2$), $\mathbb{E}[x \mid y]$ tend vers : (A) 2 ; (B) $1{,}9$ ; (C) $2{,}6$, la moyenne des relevés (une lettre) ;
- b) pour $y = 1{,}475$ (donc $\frac ya = 2{,}95$, à égale distance de $2{,}3$ et de $3{,}6$) : (A) $2{,}95$ ; (B) $2{,}3$ ; (C) $3{,}6$ (une lettre) ;
- c) pour $y = 1$ et un bruit énorme, $b = 50$ : (A) $1{,}9$ ; (B) 2 ; (C) $2{,}6$ (une lettre).

Puis, une fois tes prédictions écrites et vérifiées, mesure avec `posterior_mean_mixture` (les cinq relevés comme des points : des variances nulles) :
- d) $\mathbb{E}[x \mid y = 1]$ pour $b = 1$, $0{,}3$, $0{,}1$, $0{,}03$ et $0{,}01$, dans cet ordre (une liste de 5 nombres) ;
- e) la même chose pour $y = 1{,}475$ (une liste de 5 nombres) ;
- f) $\mathbb{E}[x \mid y = 1]$ pour $b = 50$ (un nombre).

La vérification de l'expérience ne s'exécute qu'une fois tes trois prédictions écrites.

In [ ]:
RECORDS_20 = frozen([1.2, 1.9, 2.3, 3.6, 4.0])     # five heights of the sea read by a tide gauge (m), equal weights

📝 **Mon hypothèse** (à écrire **avant** d'exécuter quoi que ce soit) : …

In [ ]:
prediction_MB5_20a = ...  # a) "A", "B" or "C": for y = 1 and b -> 0, E[x | y] tends to (A) 2, (B) 1.9, (C) 2.6
prediction_MB5_20b = ...  # b) "A", "B" or "C": for y = 1.475 and b -> 0, (A) 2.95, (B) 2.3, (C) 3.6
prediction_MB5_20c = ...  # c) "A", "B" or "C": for y = 1 and b = 50, (A) 1.9, (B) 2, (C) 2.6

In [ ]:
for letter, prediction in zip("abc", [prediction_MB5_20a, prediction_MB5_20b, prediction_MB5_20c]):
    if filled(prediction) and not isinstance(prediction, str):      # a number, a list...: not yet a prediction
        print(f'❌ Ex MB5.20{letter} : une prédiction est une lettre, "A", "B" ou "C", reçu {shown(prediction)}')
    else:
        wbm.check(f"MB5.20{letter}", prediction)

**L'expérience.** Écris-la une fois tes trois prédictions vérifiées ; sa vérification ne s'exécute qu'à cette condition.

In [ ]:
answer_MB5_20d = answer_MB5_20e = answer_MB5_20f = ...
with wbm.attempt("MB5.20"):          # your posterior_mean_mixture: ⏳ while it is not written
    answer_MB5_20d = ...  # d) a list of 5 numbers, as computed: E[x | y = 1] for b = 1, 0.3, 0.1, 0.03, 0.01
    answer_MB5_20e = ...  # e) a list of 5 numbers, as computed: E[x | y = 1.475] for the same b
    answer_MB5_20f = ...  # f) a number, as computed: E[x | y = 1] for b = 50

In [ ]:
if any(answer is ... or answer is None for answer in [prediction_MB5_20a, prediction_MB5_20b, prediction_MB5_20c]):
    print("⏳ Écris d'abord tes trois prédictions, puis exécute à nouveau.")
else:
    check_answer("MB5.20d", answer_MB5_20d, hint=library_hint("posterior_mean_mixture"))
    check_answer("MB5.20e", answer_MB5_20e, hint=library_hint("posterior_mean_mixture"))
    check_answer("MB5.20f", answer_MB5_20f, hint=library_hint("posterior_mean_mixture"))

**Ce qu'il faut regarder** (après l'expérience) : à partir de quel $b$ le débruiteur rend-il presque exactement un relevé ? Pourquoi ne rend-il jamais 2, la valeur $\frac ya$ ? Qu'est-ce qu'un débruiteur optimal fabrique de nouveau à partir d'un nuage de points ? Les réponses sont dans `05_solutions.md`, section Notebook.

### Ex MB5.21 — Le chemin droit vers une loi gaussienne : Euler et point milieu sur le champ moyen exact, de 1 à 32 pas, la moyenne et l'écart-type obtenus contre ceux de la cible 🔬 ★★ ⏱️ 25 min
**Objectif :** suivre le champ moyen exact d'un chemin droit vers une loi gaussienne par Euler et par le point milieu, et mesurer comment l'écart-type obtenu approche celui de la cible.  
**Prérequis :** fiche §8 et §9 ; Ex MB5.9, MB5.18 · **Parcours :** C, E

Les données suivent $\mathcal{N}(4 ;\ 0{,}25)$, comme en MB5.9 (`DATA_9`), et le chemin droit est $x_s = (1 - s)\,x_0 + s\,\varepsilon$. Écris `field_21(s, x)`, le champ moyen exact $v(x, s)$, affine en $x$, pour $0 < s \le 1$ : une fonction du temps $s$ puis du point $x$ (un tableau de points), pour `euler_ode` et `midpoint_ode`, qui l'appellent `f(t, x)`. La vérification la compare au débruiteur de `posterior_mean_mixture` avec $a = 1 - s$ et $b = s$, en des points cachés.

Le notebook fournit `Z_21`, 100 000 tirages de $\mathcal{N}(0, 1)$. Transporte-les de $s = 1$ à $s = 0$ (`t0=1.0`, `t1=0.0`) par Euler, puis par le point milieu, avec $N = 1$, 2, 4, 8, 16 et 32 pas. Relève :
- a) l'écart-type (`np.std`) des points obtenus, pour Euler, puis pour le point milieu (deux listes de 6 nombres ; la vérification regarde aussi leur moyenne, `np.mean`) ;
- b) l'écart-type obtenu en transportant exactement : celui de `4 + 0.5 * Z_21` (un nombre).

In [ ]:
Z_21 = frozen(np.random.default_rng(7721).normal(size=100_000))   # 100 000 draws of N(0, 1), at s = 1

In [ ]:
def field_21(s, x):
    """The exact mean field v(x, s) of the straight path to the data N(4, 0.25), for 0 < s <= 1: the time s first, then
    the points x (an array), as euler_ode and midpoint_ode call it."""
    raise NotImplementedError("field_21 n'est pas encore écrite (04_indices.md si tu bloques)")


answer_MB5_21a = answer_MB5_21b = ...
with wbm.attempt("MB5.21"):          # your field and your euler_ode, midpoint_ode: ⏳ while they are not written
    answer_MB5_21a = ...  # a) two lists of 6 numbers, as computed: np.std of the points reached from Z_21 for N = 1, 2, 4, 8, 16, 32, Euler then midpoint
    answer_MB5_21b = ...  # b) a number, as computed: np.std(4 + 0.5 * Z_21)

In [ ]:
HIDDEN_C21 = {'calls': [(0.93, [-2.31, 0.47, 1.06, 3.41]), (0.61, [-1.13, 0.29, 2.74, 4.09, 5.91]),
           (0.27, [2.17, 3.43, 4.74]), (0.08, [3.52, 3.97, 4.33, 4.71]), (1.0, [-0.83, 0.62, 2.29])],
 'id': 'MB5.21d',
 'id_means': 'MB5.21e'}   # hidden times and points, and the sub-ID of the values expected

with checked("MB5.21"):
    results_21 = []
    report_21 = reporter(results_21, stop=False)
    got_c21, changed_c21 = [], False
    for s_c21, points_c21 in HIDDEN_C21["calls"]:
        x_c21 = np.array(points_c21)                     # non-dyadic floats: a change in place would show
        got_c21.append(as_floats(call_safely("Ex MB5.21 : field_21 (appel caché)", field_21, s_c21, x_c21)))
        changed_c21 = changed_c21 or not np.array_equal(x_c21, np.array(points_c21))
    if changed_c21:
        report_21(False, "", "field_21 modifie le tableau x reçu : calcule sur une copie")
    elif any(g is None or g.shape != np.shape(p) for g, (_, p) in zip(got_c21, HIDDEN_C21["calls"])):
        report_21(False, "", "field_21 rend un tableau de la forme de x (euler_ode lui passe un tableau de points)")
    else:
        report_21(passes(HIDDEN_C21["id"], [v for g in got_c21 for v in g.tolist()]),
                  "field_21 en des points cachés (d'autres instants, d'autres points) : le champ moyen exact",
                  "field_21 en des points cachés : ce n'est pas le champ moyen v(x, s) = (x − E[x_0 | x_s = x])/s du chemin "
                  "droit x_s = (1 − s) x_0 + s ε vers N(4 ; 0,25) (fiche §9) ; essaie-la en s = 1, où elle vaut x − 4 "
                  "(MB5.9)")

field_ok_21 = bool(results_21) and all(results_21)


def hint_21(value):
    """A targeted hint for a) and b): field_21 first, then the library."""
    if not field_ok_21:
        return "les écarts-types se calculent avec field_21 : corrige-la d'abord (ligne ❌ ci-dessus)"
    return library_hint("euler_ode et midpoint_ode")(value)


def means_21():
    """a) checks the means too: np.mean of the points reached from Z_21 with your field_21, by Euler's method then the
    midpoint method of the reference, for N = 1, 2, 4, 8, 16, 32 (None if field_21 cannot be run)."""
    try:
        return [float(np.mean(method(field_21, Z_21, 1.0, 0.0, n)[-1])) for method in (ref_gp.euler_ode,
                                                                                        ref_gp.midpoint_ode)
                for n in (1, 2, 4, 8, 16, 32)]
    except Exception:  # noqa: BLE001 - field_21 fails: its own line says why
        return None


means_c21 = means_21() if filled(answer_MB5_21a) else None
means_wrong_c21 = means_c21 is not None and not passes(HIDDEN_C21["id_means"], means_c21)
if means_wrong_c21 and wbm.check("MB5.21a", answer_MB5_21a, computed=True, quiet=True).passed:   # right spreads only
    print("❌ Ex MB5.21a : la moyenne (np.mean) des points que ta field_21 fait atteindre n'est pas celle du transport "
          "exact : un terme qui ne dépend que de s déplace tous les points sans changer leur dispersion (ligne ❌ de "
          "field_21 ci-dessus)")
else:
    check_answer("MB5.21a", answer_MB5_21a, hint=hint_21)
check_answer("MB5.21b", answer_MB5_21b, hint=hint_21)

**Ce qu'il faut regarder** : pourquoi la moyenne des points (`np.mean`) est-elle juste dès un seul pas, et l'écart-type pas ? Par combien l'écart à l'écart-type exact est-il divisé quand $N$ double, pour Euler, puis pour le point milieu ? Est-ce l'ordre que donne la fiche ? Les réponses sont dans `05_solutions.md`, section Notebook.

### Ex MB5.22 — Des lois guidées p_c^{1+w} p_u^{−w} sur une grille : une loi générale à deux bosses, une loi de classe gaussienne, et ce que w resserre et déplace 🎨 ★ ⏱️ 20 min
**Objectif :** calculer et tracer des lois guidées $p_c^{1+w}p_u^{-w}$ sur une grille, et voir ce que $w$ resserre et déplace.  
**Prérequis :** fiche §7 ; M02a (méthode des trapèzes), M05a (log-sum-exp) ; Ex MB5.7 · **Parcours :** C

La loi générale est le mélange $p_u = 0{,}55\,\mathcal{N}(-1{,}8 ;\ 0{,}5) + 0{,}45\,\mathcal{N}(1{,}6 ;\ 0{,}7)$, à deux bosses (`GENERAL_22`), et la loi de la classe est $p_c = \mathcal{N}(1{,}6 ;\ 0{,}7)$ (`CLASS_22`). Sur la grille `XS_22` (`np.linspace(-8, 8, 3201)`), écris `guided_22(xs, w, general=GENERAL_22, cls=CLASS_22)`, qui rend la densité normalisée de $p_c^{1+w}p_u^{-w}$ sur la grille `xs` reçue : calcule son logarithme, retire son maximum avant l'exponentielle, puis normalise par `np.trapezoid` (SciPy est permis, `scipy.stats.norm` et `scipy.special.logsumexp` compris) ; puis `draw_guided_22(ax, ws)`, qui trace dans les axes reçus, sur `XS_22`, $p_u$, $p_c$ et la loi guidée pour chaque $w$ de `ws`, avec une légende qui nomme chaque courbe ($p_u$, $p_c$, et la valeur de $w$ de chaque loi guidée) et des noms d'axes, et rend la liste des densités guidées tracées (une par $w$, dans l'ordre de `ws`). La fonction dessine dans les axes reçus, sans ouvrir de figure.

Sur une échelle linéaire, les trois lois guidées se superposent presque à $p_c$ : tu peux ajouter un second axe en échelle logarithmique, `ax.set_yscale("log")`, pour voir la masse quitter le côté de l'autre bosse.

La vérification appelle `draw_guided_22` avec $w = 0{,}25$, $1{,}75$ et $3{,}5$, affiche la figure, et essaie tes fonctions sur des appels cachés (d'autres $w$, dans un autre ordre ; un autre mélange, sur une autre grille) ; `guided_22` ne modifie pas la grille reçue. Relève, pour $w = 0{,}25$, $1{,}75$ et $3{,}5$, avec `np.trapezoid` sur `XS_22` :
- a) la moyenne de la loi guidée, $\int x\,d(x)\,\mathrm{d}x$ (une liste de 3 nombres) ;
- b) son écart-type, la racine de $\int (x - \text{moyenne})^2\,d(x)\,\mathrm{d}x$ (une liste de 3 nombres) ;
- c) pour $w = 3{,}5$ seulement, sa masse sur $x < 0$, le côté de l'autre bosse : `np.trapezoid(np.where(XS_22 < 0, d, 0), XS_22)`, où `d` est la densité sur la grille (un nombre).

In [ ]:
# The general law p_u, a mixture with two bumps (weights, means, variances), the law p_c of the class (mean, variance),
# and the grid
GENERAL_22 = ((0.55, 0.45), (-1.8, 1.6), (0.5, 0.7))
CLASS_22 = (1.6, 0.7)
XS_22 = frozen(np.linspace(-8, 8, 3201))

In [ ]:
def guided_22(xs, w, general=GENERAL_22, cls=CLASS_22):
    """The density on the grid xs of the guided law, proportional to p_c^(1 + w) p_u^(-w), normalized by np.trapezoid:
    p_u is the mixture general = (weights, means, variances), p_c the Gaussian cls = (mean, variance)."""
    raise NotImplementedError("guided_22 n'est pas encore écrite (04_indices.md si tu bloques)")


def draw_guided_22(ax, ws):
    """In the axes ax, on the grid XS_22: p_u, p_c and the guided law for each w of ws, with a legend that names each
    curve and axis labels. Returns the list of the guided densities drawn (one per w, in the order of ws). Opens no
    figure."""
    raise NotImplementedError("draw_guided_22 n'est pas encore écrite (04_indices.md si tu bloques)")


answer_MB5_22a = answer_MB5_22b = answer_MB5_22c = ...
with wbm.attempt("MB5.22"):          # your functions: ⏳ while they are not written
    answer_MB5_22a = ...  # a) a list of 3 numbers, as computed: the means of the guided laws for w = 0.25, 1.75, 3.5
    answer_MB5_22b = ...  # b) a list of 3 numbers, as computed: their standard deviations
    answer_MB5_22c = ...  # c) a number, as computed: for w = 3.5, the mass on x < 0

In [ ]:
HIDDEN_C22 = {'idx': [0, 32, 64, 96, 128, 160, 192, 224, 256, 288, 320, 352, 384, 416, 448, 480, 544, 576, 608, 640, 704,
         736, 768, 800, 832, 896, 928, 960, 992, 1024, 1056, 1088, 1120, 1216, 1248, 1312, 1376, 1440, 1504,
         1568, 1600, 1632, 1664, 1728, 1856, 1888, 1920, 2048, 2112, 2176, 2336, 2400, 2528, 2688, 2720, 2752,
         2784, 2816, 2848, 2880, 2912, 2944, 2976, 3008, 3040, 3072, 3104, 3136, 3168, 3200],
 'ws': [2.8, 0.45, 1.1],
 'grid': (-7.0, 6.0, 2601),
 'general': ((0.47, 0.53), (-2.3, 0.9), (0.6, 0.45)),
 'cls': (0.9, 0.45),
 'other_ws': [0.9, 2.2],
 'idx_other': [0, 26, 52, 78, 104, 130, 156, 182, 208, 234, 260, 286, 312, 338, 364, 390, 416, 442, 468, 494,
               520, 546, 572, 598, 624, 650, 676, 702, 728, 754, 780, 806, 832, 858, 884, 910, 936, 962, 988,
               1014, 1040, 1066, 1092, 1118, 1144, 1170, 1222, 1248, 1274, 1326, 1352, 1430, 1456, 1482, 1508,
               1560, 1586, 1664, 1690, 1716, 1742, 1768, 1794, 1898, 1924, 1950, 1976, 2028, 2054, 2080, 2106,
               2184, 2210, 2236, 2314, 2340, 2366, 2392, 2418, 2444, 2470, 2496, 2522, 2548, 2574, 2600],
 'id_main': 'MB5.22d',
 'id_hidden': 'MB5.22e',
 'id_other': 'MB5.22f',
 'id_u': 'MB5.22g',
 'id_c': 'MB5.22h'}   # hidden calls, the points of the grids compared, and the sub-IDs of the values expected

_GENERAL_NAME_C22 = re.compile(r"p_?\{?u\}?|\bpu\b|génér|gener|incond|uncond", re.IGNORECASE)
_CLASS_NAME_C22 = re.compile(r"p_?\{?c\}?|\bpc\b|classe|\bclass|\bcond", re.IGNORECASE)


def label_w_c22(label):
    """The value of w written in a label of the legend (« w = 0,25 », « $w=1.75$ », « 3,5 »), or None."""
    text = str(label)
    if not text or text.startswith("_"):
        return None
    text = text.replace("{,}", ".").replace(",", ".")
    found = re.search(r"w\s*[=:]\s*\$?\s*([-+]?\d+(?:\.\d+)?)", text)
    if found:
        return float(found.group(1))
    text = re.sub(r"1\s*\+\s*w|[-−]\s*w|p_?\{?[cu]\}?|\^\{[^}]*\}", " ", text)
    found = re.search(r"(?<![\w.])(\d+(?:\.\d+)?)", text)
    return float(found.group(1)) if found else None


def density_c22(label, xs, w, **kwargs):
    """guided_22(xs, w, **kwargs) on a copy of xs, as a float array of the shape of xs (one ❌ line otherwise)."""
    grid = np.array(xs, dtype=float)
    got = as_floats(call_safely(label, guided_22, grid, w, **kwargs))
    if not np.array_equal(grid, np.asarray(xs, dtype=float)):
        print(f"❌ {label} : guided_22 modifie la grille xs reçue : calcule sur une copie")
        raise StopCheck
    if got is None or got.shape != grid.shape:
        print(f"❌ {label} : guided_22 rend un tableau de la forme de xs, reçu {shown(got)}")
        raise StopCheck
    return got


def figure_why_c22(ax, ws, curves, sub_id):
    """What is wrong in a call of draw_guided_22 (French phrases)."""
    idx = HIDDEN_C22["idx"]
    if not isinstance(curves, (list, tuple)) or len(curves) != len(ws):
        return [f"draw_guided_22 doit rendre la liste des {len(ws)} densités guidées tracées (une par w), reçu "
                f"{shown(curves)}"]
    curves = [as_floats(c) for c in curves]
    if any(c is None or c.shape != XS_22.shape for c in curves):
        return ["chaque densité rendue par draw_guided_22 doit avoir une valeur par point de XS_22"]
    problems = []
    if not passes(sub_id, [c[idx].tolist() for c in curves]):
        problems.append("les densités rendues doivent être celles de la loi guidée pour chaque w reçu, dans l'ordre reçu")
    lines = [line_with(ax, XS_22, c) for c in curves]
    if any(line is None for line in lines):
        problems.append("chaque loi guidée doit être tracée en fonction des points de XS_22")
    on_grid = [line for line in ax.lines if np.asarray(line.get_xdata(), dtype=float).shape == XS_22.shape
               and np.allclose(np.asarray(line.get_xdata(), dtype=float), XS_22)]
    general = [line for line in on_grid if passes(HIDDEN_C22["id_u"], np.asarray(line.get_ydata(), dtype=float)[idx].tolist())]
    cls = [line for line in on_grid if passes(HIDDEN_C22["id_c"], np.asarray(line.get_ydata(), dtype=float)[idx].tolist())]
    if not general or not cls:
        problems.append("il faut tracer aussi p_u et p_c, sur XS_22")
    if ax.get_legend() is None:
        problems.append("il faut une légende (label=…, puis ax.legend())")
    else:
        if (general and not _GENERAL_NAME_C22.search(general[0].get_label())) or (
                cls and not _CLASS_NAME_C22.search(cls[0].get_label())):
            problems.append("la légende doit nommer p_u et p_c")
        if any(line is not None and not math.isclose(label_w_c22(line.get_label()) or math.nan, w)
               for line, w in zip(lines, ws)):
            problems.append("la légende doit nommer le w de chaque loi guidée (label=f\"w = {w}\")")
    if not ax.get_xlabel() or not ax.get_ylabel():
        problems.append("il faut des noms d'axes")
    return problems


with checked("MB5.22"):
    results_22 = []
    report_22 = reporter(results_22, stop=False)
    idx_c22 = HIDDEN_C22["idx"]
    main_c22 = [density_c22("Ex MB5.22 : guided_22", XS_22, w) for w in (0.25, 1.75, 3.5)]
    report_22(passes(HIDDEN_C22["id_main"], [d[idx_c22].tolist() for d in main_c22]),
              "guided_22 (w = 0,25, 1,75 et 3,5, sur XS_22) : les trois densités",
              "guided_22 (w = 0,25, 1,75 et 3,5, sur XS_22) : ce ne sont pas les densités de p_c^{1+w} p_u^{−w}, "
              "normalisées par np.trapezoid sur la grille reçue")
    hidden_c22 = [density_c22("Ex MB5.22 : guided_22 (appel caché)", XS_22, w) for w in HIDDEN_C22["ws"]]
    grid_c22 = np.linspace(*HIDDEN_C22["grid"])
    other_c22 = [density_c22("Ex MB5.22 : guided_22 (appel caché)", grid_c22, w, general=HIDDEN_C22["general"],
                             cls=HIDDEN_C22["cls"]) for w in HIDDEN_C22["other_ws"]]
    report_22(passes(HIDDEN_C22["id_hidden"], [d[idx_c22].tolist() for d in hidden_c22])
              and passes(HIDDEN_C22["id_other"], [d[HIDDEN_C22["idx_other"]].tolist() for d in other_c22]),
              "des appels cachés de guided_22 (d'autres w, dans un autre ordre ; un autre mélange, sur une autre grille)",
              "des appels cachés de guided_22 (d'autres w, dans un autre ordre ; un autre mélange, sur une autre grille) : "
              "ta fonction doit se servir des xs, w, general et cls reçus")
    fig_22, ax_22 = plt.subplots(figsize=(9, 4.5))
    try:
        drawn_22 = []
        opened_22 = drawn(lambda: drawn_22.append(call_safely("Ex MB5.22 : draw_guided_22", draw_guided_22, ax_22,
                                                              (0.25, 1.75, 3.5))))
        problems_22 = figure_why_c22(ax_22, (0.25, 1.75, 3.5), drawn_22[0], HIDDEN_C22["id_main"])
        report_22(not problems_22, "draw_guided_22 (w = 0,25, 1,75 et 3,5) : p_u, p_c, les trois lois guidées, la légende "
                  "et les noms d'axes", "draw_guided_22 : " + " ; ".join(problems_22))
        fig_c22, axes_c22 = plt.subplots(1, 3)          # the 3rd axes is the current one: plt.plot would draw there
        try:
            hidden_drawn_c22 = []
            opened_22 += drawn(lambda: hidden_drawn_c22.append(call_safely(
                "Ex MB5.22 : draw_guided_22 (appel caché)", draw_guided_22, axes_c22[0], tuple(HIDDEN_C22["ws"]))))
            problems_c22 = figure_why_c22(axes_c22[0], HIDDEN_C22["ws"], hidden_drawn_c22[0], HIDDEN_C22["id_hidden"])
            elsewhere_22 = bool(axes_c22[2].lines or axes_c22[2].collections or axes_c22[2].get_legend())
        finally:
            plt.close(fig_c22)
        report_22(not problems_c22, "un appel caché de draw_guided_22 (d'autres w, dans un autre ordre) est juste aussi",
                  "un appel caché de draw_guided_22 (d'autres w, dans un autre ordre) : " + " ; ".join(problems_c22))
        report_22(not opened_22 and not elsewhere_22, "draw_guided_22 dessine dans les axes reçus",
                  "draw_guided_22 doit dessiner dans les axes reçus (ax.plot, ax.legend…), sans ouvrir de figure ni "
                  "dessiner dans l'axe courant (plt.plot, plt.legend…)")
    except BaseException:
        plt.close(fig_22)
        raise
    fig_22.tight_layout()
    plt.show()
    verdict("MB5.22", all(results_22) and len(results_22) == 5, "ta loi guidée et ta figure sont justes, sur les appels du "
            "notebook et sur des appels cachés.", "relis les lignes ❌ ci-dessus.")

guided_ok_22 = bool(results_22) and all(results_22[:2])


def hint_22(value):
    """A targeted hint for a) to c): guided_22 first."""
    return None if guided_ok_22 else "les moments se calculent sur les densités de guided_22 : corrige-la d'abord"


check_answer("MB5.22a", answer_MB5_22a, hint=hint_22)
check_answer("MB5.22b", answer_MB5_22b, hint=hint_22)
check_answer("MB5.22c", answer_MB5_22c, hint=hint_22)

**Ce qu'il faut regarder** : que devient, pour $w = 3{,}5$, la masse du côté de l'autre bosse, comparée à celle de $p_c$ ? Pourquoi la moyenne s'écarte-t-elle de $1{,}6$, et de quel côté ? Pourquoi ces lois existent-elles pour tout $w \ge 0$ ? Les réponses sont dans `05_solutions.md`, section Notebook.

## Partie C · Déboguer, tester

Fiche §2, §5 et §8. Trois erreurs classiques, qui passent une vérification mal choisie ; trois tests de propriété d'une méthode d'intégration. Les méthodes cachées s'appuient sur la référence (`ref_gp`), pour ne pas dépendre de ton code.

### Ex MB5.23 — Chasse à l'erreur : un pont aux poids inversés, un débruiteur qui oublie le bruit dans ses responsabilités, et un Euler dont le temps n'avance pas 🐛 ★ ⏱️ 20 min
**Objectif :** reconnaître trois erreurs classiques (des poids inversés dans un pont, le bruit oublié dans les responsabilités, un temps qui n'avance pas), qui passent une vérification mal choisie, puis les corriger.  
**Prérequis :** fiche §2, §5 et §8 ; Ex MB5.3, MB5.17, MB5.18 · **Parcours :** C, E

Un collègue a écrit trois fonctions, chacune accompagnée d'une vérification qui passe (cellule fournie) :
1. `bridge_bad(x0, xT, Vs, VT)` doit rendre la moyenne et la variance du pont d'une marche aléatoire ; elle met le poids $\frac{V_s}{V_T}$ sur $x_0$ au lieu de $x_T$. Il l'a vérifiée au milieu d'une marche de pas égaux.
2. `denoiser_bad(y, a, b, weights, means, variances)` doit rendre $\mathbb{E}[x \mid y]$ pour un mélange ; elle calcule les responsabilités avec la variance $v_k$ du signal seule, au lieu de $a^2v_k + b^2$. Il l'a vérifiée sur un mélange à une seule composante.
3. `euler_bad(f, x0, t0, t1, n_steps)` doit suivre la méthode d'Euler ; elle lit le champ en $t_0$ à chaque pas. Il l'a vérifiée sur $x' = -\frac x3$.

À faire : trouve et corrige chaque erreur dans le code du collègue lui-même : les fonctions corrigées s'appellent `bridge_fixed`, `denoiser_fixed` et `euler_fixed`, avec les mêmes arguments, et n'appellent **pas** `mymath` (la vérification le contrôle). Puis réponds :
- a) pour le phoque de MB5.3 (`SEAL_3`, cellule de données), la moyenne de la position à 11 h par la fonction corrigée, puis par la fonction fautive (une liste de 2 nombres) ;
- b) pour le mélange $0{,}4\,\mathcal{N}(-1 ;\ 0{,}3) + 0{,}6\,\mathcal{N}(2{,}5 ;\ 0{,}6)$ mesuré par $y = 1{,}2\,x + 0{,}9\,n$, $\mathbb{E}[x \mid y]$ pour $y = 0{,}4$ et $1{,}4$, par la fonction corrigée, puis par la fonction fautive (une liste de 4 nombres) ;
- c) $x' = t - x$, $x(0) = 1$, sur $[0, 2]$ en six pas : le point atteint en $t = 2$ par la fonction corrigée, puis par la fonction fautive (une liste de 2 nombres).

Dans ta copie : pour chacune des trois erreurs, pourquoi la vérification du collègue ne la voyait pas. La vérification essaie aussi tes fonctions corrigées sur des cas cachés : d'autres ponts, des mélanges de deux composantes ou plus, des champs qui dépendent du temps, et des tableaux qu'elles ne doivent pas modifier.

In [ ]:
# The colleague's code
def bridge_bad(x0, xT, Vs, VT):
    """The mean and the variance of a random walk at the time of cumulated variance Vs, given its value x0 at the start
    (cumulated variance 0) and its value xT at the time of cumulated variance VT (the bridge)."""
    weight = Vs / VT
    return weight * x0 + (1 - weight) * xT, Vs * (VT - Vs) / VT


def denoiser_bad(y, a, b, weights, means, variances):
    """E[x | y] for the measurement y = a x + b n, when x follows the mixture of the Gaussians N(means[k], variances[k])
    of weights weights[k] (variances > 0)."""
    y = np.asarray(y, dtype=float)
    w, m, v = (np.asarray(part, dtype=float) for part in (weights, means, variances))
    gap = y.reshape(-1, 1) - a * m                                       # y - a mu_k, one row per measurement
    logs = np.log(w) - 0.5 * np.log(2 * np.pi * v) - gap ** 2 / (2 * v)  # the responsibilities, by logarithms
    resp = np.exp(logs - logs.max(axis=1, keepdims=True))
    resp = resp / resp.sum(axis=1, keepdims=True)
    inside = m + a * v / (a ** 2 * v + b ** 2) * gap                     # E[x | y, k]
    return (resp * inside).sum(axis=1).reshape(y.shape)


def euler_bad(f, x0, t0, t1, n_steps):
    """The points of Euler's method for x' = f(t, x), from x(t0) = x0 to the final time t1, in n_steps steps."""
    h = (t1 - t0) / n_steps
    x = np.array(x0, dtype=float)
    points = [x.copy()]
    for k in range(n_steps):
        x = x + h * f(t0, x)
        points.append(x.copy())
    return np.array(points)


# The colleague's checks: all pass
mean_23, var_23 = bridge_bad(1.0, 7.0, 3.0, 6.0)          # halfway through a walk of equal steps
print("(1)", math.isclose(mean_23, 4.0) and math.isclose(var_23, 1.5),
      "(attendu True : à mi-chemin, la moyenne des deux valeurs et la variance V_s(V_T − V_s)/V_T)")
one_23 = float(denoiser_bad(2.0, 1.5, 0.7, [1.0], [0.4], [0.9]))   # one Gaussian: the linear Gaussian model of M25a
print("(2)", math.isclose(one_23, 0.4 + 1.5 * 0.9 * (2.0 - 1.5 * 0.4) / (1.5 ** 2 * 0.9 + 0.7 ** 2)),
      "(attendu True : une seule gaussienne, la formule de M25a)")
path_23 = euler_bad(lambda t, x: -x / 3, 3.0, 0.0, 2.0, 5)          # each step multiplies x by 1 - h/3
print("(3)", bool(np.allclose(path_23, 3.0 * (1 - 0.4 / 3) ** np.arange(6))),
      "(attendu True : chaque pas multiplie x par 1 − h/3)")

In [ ]:
def bridge_fixed(x0, xT, Vs, VT):
    """The mean and the variance of a random walk at the time of cumulated variance Vs, given its value x0 at the start
    (cumulated variance 0) and its value xT at the time of cumulated variance VT (the bridge)."""
    raise NotImplementedError("bridge_fixed n'est pas encore écrite (04_indices.md si tu bloques)")


def denoiser_fixed(y, a, b, weights, means, variances):
    """E[x | y] for the measurement y = a x + b n, when x follows the mixture of the Gaussians N(means[k], variances[k])
    of weights weights[k] (variances > 0)."""
    raise NotImplementedError("denoiser_fixed n'est pas encore écrite (04_indices.md si tu bloques)")


def euler_fixed(f, x0, t0, t1, n_steps):
    """The points of Euler's method for x' = f(t, x), from x(t0) = x0 to the final time t1, in n_steps steps."""
    raise NotImplementedError("euler_fixed n'est pas encore écrite (04_indices.md si tu bloques)")


answer_MB5_23a = answer_MB5_23b = answer_MB5_23c = ...
with wbm.attempt("MB5.23"):          # your three functions: ⏳ while they are not written
    answer_MB5_23a = ...  # a) a list of 2 numbers, as computed: the mean at 11 h by bridge_fixed, then by the WRONG bridge_bad
    answer_MB5_23b = ...  # b) a list of 4 numbers, as computed: denoiser_fixed at y = 0.4 and 1.4, then the WRONG denoiser_bad
    answer_MB5_23c = ...  # c) a list of 2 numbers, as computed: the point at t = 2 by euler_fixed, then by the WRONG euler_bad

In [ ]:
for letter, answer in zip("abc", [answer_MB5_23a, answer_MB5_23b, answer_MB5_23c]):
    check_answer(f"MB5.23{letter}", answer)

CASES_C23 = {'bridges': [[-2.3, 4.1, 0.7, 5.3], [6.2, 1.9, 4.35, 4.9], [0.35, -1.15, 2.65, 3.1], [10.5, 12.25, 0.9, 7.8]],
 'mixtures': [[[-1.1, 0.35, 2.05, 3.3], 1.3, 0.6, [0.27, 0.73], [-0.6, 1.9], [0.45, 0.2]],
              [[-1.7, -0.2, 0.9, 2.6], 0.85, 1.15, [0.2, 0.5, 0.3], [-2.1, 0.4, 2.7], [0.15, 0.6, 0.35]],
              [[[-1.3, 0.45], [1.15, -2.2]], -0.75, 0.55, [0.1, 0.25, 0.4, 0.25], [-3.2, -1.0, 0.7, 2.9],
               [0.3, 0.12, 0.5, 0.25]]],
 'fields': [['wave', [0.3, -1.7, 2.45], 0.0, 2.6, 7], ['ramp', [1.15, 0.05], 0.5, 3.1, 5],
            ['wave', [0.9, -0.35], 2.2, -0.4, 6]],
 'id': 'MB5.23d'}   # the hidden bridges, mixtures and fields, and the sub-ID of the bridges expected

results_23, parts_23 = [], []
MYMATH_C23 = list(getattr(mymath, "__path__", [])) + list(getattr(REF, "__path__", []))   # the folders of mymath
TWINS_C23 = set()                                  # the functions of gaussian_paths.py (yours, the reference's)
for _folder_c23 in MYMATH_C23:
    _file_c23 = os.path.join(_folder_c23, "gaussian_paths.py")
    if os.path.exists(_file_c23):
        TWINS_C23 |= code_keys_of_file(_file_c23)
FIELDS_C23 = {"wave": lambda t, x: 0.75 * np.cos(1.3 * t) - 0.45 * x,     # hidden fields that depend on the time
              "ramp": lambda t, x: 0.6 * t - 0.25 * x * x}


def report_23(ok, success, failure):
    results_23.append(bool(ok))
    print(("✅ " + success) if ok else ("❌ " + failure))


def call_23(part, name, function, *args):
    """function(*args), without mymath; an error of the learner's code becomes one ❌ line that names the function."""
    try:
        with code_from(MYMATH_C23, twins=TWINS_C23) as seen:
            got = function(*args)
    except (NotImplementedError, MymathMissingError):
        raise
    except Exception as exc:  # noqa: BLE001 - the learner's error, reported with the name of the function
        report_23(False, "", f"({part}) {name} : ton code lève {type(exc).__name__} : {str(exc)[:160]}")
        parts_23.append(part)
        raise StopCheck from None
    if seen:
        report_23(False, "", f"({part}) {name} appelle mymath ({seen[0]}) : l'énoncé demande la correction sans mymath, "
                  "en corrigeant le code du collègue")
        parts_23.append(part)
        raise StopCheck
    return got


def unchanged_23(arrays, copies):
    return all(np.array_equal(a, c) for a, c in zip(arrays, copies))


with checked("MB5.23"):                                       # (1) bridge_fixed
    got_c23 = []
    for case_c23 in CASES_C23["bridges"]:
        pair_c23 = as_floats(call_23(1, "bridge_fixed", bridge_fixed, *case_c23))
        got_c23.append(pair_c23.tolist() if pair_c23 is not None and pair_c23.shape == (2,) else None)
    if any(pair is None for pair in got_c23):
        why_c23 = "bridge_fixed rend un couple (moyenne, variance) de deux nombres"
    elif passes(CASES_C23["id"], got_c23):
        why_c23 = ""
    elif all(math.isclose(pair[0], bridge_bad(*case)[0], rel_tol=1e-9, abs_tol=1e-12)
             for pair, case in zip(got_c23, CASES_C23["bridges"])):
        why_c23 = "ta correction met encore le poids V_s/V_T sur x_0 : il va avec x_T, la valeur à l'instant de V_T"
    else:
        why_c23 = ("sur d'autres ponts, ce n'est pas la moyenne x_0 + (V_s/V_T)(x_T − x_0) et la variance "
                   "V_s(V_T − V_s)/V_T du pont d'une marche aléatoire")
    report_23(not why_c23, "(1) bridge_fixed : d'autres ponts, dont des instants proches de la fin",
              f"(1) bridge_fixed : {why_c23}")
    parts_23.append(1)

with checked("MB5.23"):                                       # (2) denoiser_fixed, compared with the reference
    wrong_c23 = []
    for y_c23, a_c23, b_c23, w_c23, m_c23, v_c23 in CASES_C23["mixtures"]:
        inputs_c23 = [np.array(y_c23), np.array(w_c23), np.array(m_c23), np.array(v_c23)]   # non-dyadic floats
        copies_c23 = [x.copy() for x in inputs_c23]
        got_c23 = as_floats(call_23(2, "denoiser_fixed", denoiser_fixed, inputs_c23[0], a_c23, b_c23, *inputs_c23[1:]))
        want_c23 = ref_gp.posterior_mean_mixture(copies_c23[0], a_c23, b_c23, *copies_c23[1:])
        if not unchanged_23(inputs_c23, copies_c23):
            wrong_c23.append("ta fonction modifie un tableau reçu : travaille sur une copie")
        elif got_c23 is None or got_c23.shape != want_c23.shape:
            wrong_c23.append("denoiser_fixed rend un tableau de la forme de y")
        elif matches(got_c23, want_c23, rtol=1e-9, atol=1e-9):
            continue
        elif matches(got_c23, denoiser_bad(copies_c23[0], a_c23, b_c23, *copies_c23[1:]), rtol=1e-9, atol=1e-9):
            wrong_c23.append("ta correction calcule encore les responsabilités avec la variance s_k² du signal seule : la "
                             "loi de y sachant la composante k a la variance a² s_k² + b²")
        else:
            wrong_c23.append("sur des mélanges de deux composantes ou plus, ce n'est pas E[x | y] : les responsabilités "
                             "viennent de la loi N(a μ_k, a² s_k² + b²) de y sachant la composante k")
    report_23(not wrong_c23, "(2) denoiser_fixed : des mélanges cachés de deux à quatre composantes, des mesures en "
              "tableau", "(2) denoiser_fixed : " + " ; ".join(dict.fromkeys(wrong_c23)))
    parts_23.append(2)

with checked("MB5.23"):                                       # (3) euler_fixed, compared with the reference
    wrong_c23 = []
    for name_c23, x0_c23, t0_c23, t1_c23, n_c23 in CASES_C23["fields"]:
        start_c23 = np.array(x0_c23)                          # non-dyadic floats: a change in place would show
        got_c23 = as_floats(call_23(3, "euler_fixed", euler_fixed, FIELDS_C23[name_c23], start_c23, t0_c23, t1_c23,
                                    n_c23))
        want_c23 = ref_gp.euler_ode(FIELDS_C23[name_c23], np.array(x0_c23), t0_c23, t1_c23, n_c23)
        if not np.array_equal(start_c23, np.array(x0_c23)):
            wrong_c23.append("ta fonction modifie le point de départ reçu : travaille sur une copie")
        elif got_c23 is None or got_c23.shape != want_c23.shape:
            wrong_c23.append("euler_fixed rend les N + 1 points, de forme (N + 1,) + la forme de x0")
        elif matches(got_c23, want_c23, rtol=1e-9, atol=1e-9):
            continue
        elif matches(got_c23, euler_bad(FIELDS_C23[name_c23], np.array(x0_c23), t0_c23, t1_c23, n_c23), rtol=1e-9,
                     atol=1e-9):
            wrong_c23.append("le temps du champ n'avance pas encore : le pas k lit la vitesse en t_k = t_0 + k h")
        else:
            wrong_c23.append("sur des champs qui dépendent du temps, ce ne sont pas les points de la méthode d'Euler, "
                             "x_{k+1} = x_k + h f(t_k, x_k)")
    report_23(not wrong_c23, "(3) euler_fixed : des champs cachés qui dépendent du temps, dans les deux sens du temps",
              "(3) euler_fixed : " + " ; ".join(dict.fromkeys(wrong_c23)))
    parts_23.append(3)

if len(set(parts_23)) == 3:
    verdict("MB5.23", all(results_23), "les trois fonctions corrigées passent tous les cas.", "relis les lignes ❌ ci-dessus.")

**Après la vérification**, dans ta copie : pour chacune des trois erreurs, pourquoi la vérification du collègue ne la voyait-elle pas, et quelle vérification l'aurait vue ?

### Ex MB5.24 — Un test de propriété : Euler exact pour une vitesse constante, le point milieu exact pour une vitesse affine en t, et l'erreur d'Euler divisée par deux avec le pas 🛠️ ★ ⏱️ 30 min
**Objectif :** écrire trois tests de propriété d'une méthode d'intégration, et voir ce que chacun attrape.  
**Prérequis :** fiche §8 ; M02a (sommes de Riemann, point milieu) ; Ex MB5.18 · **Parcours :** C

Écris trois fonctions qui testent une méthode d'intégration `method(f, x0, t0, t1, n_steps)` qu'on leur passe (de la forme de `euler_ode` : elle rend les $N + 1$ points), et rendent `True` si la propriété tient, `False` sinon :
- `constant_is_exact_24(method)` vérifie qu'avec un champ constant (plusieurs valeurs, plusieurs nombres de pas, avec $t_0 < t_N$), **chaque** point rendu est sur la droite exacte, à $10^{-9}$ près en valeur absolue (plus fin que la tolérance par défaut de `np.allclose`, `rtol=1e-5` et `atol=1e-8`) ;
- `affine_is_exact_24(method)` vérifie de même qu'avec un champ $f(t, x) = \kappa_0 + \kappa_1t$, chaque point rendu est sur la parabole exacte, à $10^{-9}$ près en valeur absolue ;
- `error_halves_24(method)` vérifie que, pour un champ qui dépend de $t$ et de $x$ dont tu connais la solution exacte, l'erreur au temps final est divisée par un nombre entre $1{,}8$ et $2{,}2$ quand le nombre de pas passe de $N$ à $2N$ (avec $N$ assez grand).

La vérification lance tes trois tests sur `euler_ode` et `midpoint_ode` de la référence (`ref_gp.euler_ode`, `ref_gp.midpoint_ode`), sur deux autres écritures justes de ces méthodes, puis sur des méthodes fausses cachées, passées sous un nom neutre (un test juge ce que fait la méthode, pas son nom), une par clause de l'énoncé, que chaque test doit refuser quand sa propriété est fausse. Relève :
- a) les résultats de tes trois tests sur `euler_ode`, puis sur `midpoint_ode` de la référence (deux listes de 3 booléens, dans l'ordre constant, affine, erreur).

In [ ]:
def constant_is_exact_24(method):
    """True when, for constant velocity fields (several values, several numbers of steps, t0 < t1), every point returned
    by method(f, x0, t0, t1, n_steps) is on the exact line, up to 1e-9; False otherwise."""
    raise NotImplementedError("constant_is_exact_24 n'est pas encore écrite (04_indices.md si tu bloques)")


def affine_is_exact_24(method):
    """True when, for fields f(t, x) = k0 + k1 t (several fields, several numbers of steps, t0 < t1), every point
    returned by method is on the exact parabola, up to 1e-9; False otherwise."""
    raise NotImplementedError("affine_is_exact_24 n'est pas encore écrite (04_indices.md si tu bloques)")


def error_halves_24(method):
    """True when, for a field that depends on t and on x and whose exact solution you know, the error of method at the
    final time is divided by a number between 1.8 and 2.2 when the number of steps goes from N to 2N (N large enough);
    False otherwise."""
    raise NotImplementedError("error_halves_24 n'est pas encore écrite (04_indices.md si tu bloques)")


answer_MB5_24a = ...
with wbm.attempt("MB5.24"):          # your three tests: ⏳ while they are not written
    answer_MB5_24a = ...  # a) two lists of 3 booleans: your three tests on ref_gp.euler_ode, then on ref_gp.midpoint_ode

In [ ]:
for letter, answer in zip("a", [answer_MB5_24a]):
    check_answer(f"MB5.24{letter}", answer)

def euler_twin_c24(f, x0, t0, t1, n_steps):
    """RIGHT: Euler's method written another way (the times accumulated step after step, the points as the start plus
    the sum of the steps): rounding errors of about 1e-16."""
    h = (t1 - t0) / n_steps
    x, t, moves = np.array(x0, dtype=float), t0, []
    for _ in range(n_steps):
        moves.append(h * np.asarray(f(t, x), dtype=float))
        x, t = x + moves[-1], t + h
    return np.array([np.array(x0, dtype=float)] + [np.array(x0, dtype=float) + s for s in np.cumsum(moves, axis=0)])


def midpoint_twin_c24(f, x0, t0, t1, n_steps):
    """RIGHT: the midpoint method written another way (the times accumulated step after step)."""
    h = (t1 - t0) / n_steps
    x, t = np.array(x0, dtype=float), t0
    out = [x]
    for _ in range(n_steps):
        x = x + h * np.asarray(f(t + 0.5 * h, x + 0.5 * h * np.asarray(f(t, x), dtype=float)), dtype=float)
        t = t + h
        out.append(x)
    return np.array(out)


def middle_points_off_c24(f, x0, t0, t1, n_steps):
    """WRONG: the points of the midpoint method, those in between moved by 1e-3 k (N - k) (the first and the last are
    right)."""
    points = ref_gp.midpoint_ode(f, x0, t0, t1, n_steps)
    k = np.arange(n_steps + 1, dtype=float).reshape((-1,) + (1,) * (points.ndim - 1))
    return points + 1e-3 * k * (n_steps - k)


def one_step_only_c24(f, x0, t0, t1, n_steps):
    """WRONG from two steps on: the midpoint method with steps of length (t1 - t0) / N² (right for one step)."""
    return ref_gp.midpoint_ode(f, x0, t0, t0 + (t1 - t0) / n_steps, n_steps)


def unit_speed_c24(f, x0, t0, t1, n_steps):
    """WRONG unless the velocity is 0 or 1: Euler's method with the velocity squared."""
    return ref_gp.euler_ode(lambda t, x: np.asarray(f(t, x), dtype=float) ** 2, x0, t0, t1, n_steps)


def unit_slope_c24(f, x0, t0, t1, n_steps):
    """WRONG unless the field grows by 1 per unit of time: Euler's method plus h²/2 at each step."""
    h = (t1 - t0) / n_steps
    return ref_gp.euler_ode(lambda t, x: np.asarray(f(t, x), dtype=float) + h / 2, x0, t0, t1, n_steps)


def slightly_off_c24(f, x0, t0, t1, n_steps):
    """WRONG by more than 1e-9, but less than np.allclose's default tolerance (1e-8 + 1e-5 |x|): the points of the
    midpoint method, each point after the first moved by 5e-9 + 5e-6 |x_k|."""
    points = ref_gp.midpoint_ode(f, x0, t0, t1, n_steps)
    moved = points + 5e-9 + 5e-6 * np.abs(points)
    moved[0] = points[0]
    return moved


def frozen_time_c24(f, x0, t0, t1, n_steps):
    """WRONG for a field that depends on the time: Euler's method reading the field at t0 at each step."""
    return ref_gp.euler_ode(lambda t, x: f(t0, x), x0, t0, t1, n_steps)


def frozen_point_c24(f, x0, t0, t1, n_steps):
    """WRONG for a field that depends on x: Euler's method reading the field at the start x0 at each step."""
    start = np.array(x0, dtype=float)
    return ref_gp.euler_ode(lambda t, x: f(t, start.copy()), x0, t0, t1, n_steps)


def halfway_c24(f, x0, t0, t1, n_steps):
    """WRONG: Euler's method that stops halfway, at (t0 + t1) / 2 (its error does not tend to 0)."""
    return ref_gp.euler_ode(f, x0, t0, (t0 + t1) / 2, n_steps)


WRONG_C24 = [   # a hidden wrong method, the test that must refuse it, and the clause of the statement it tries
    (middle_points_off_c24, "constant_is_exact_24", "aux points intermédiaires (le premier et le dernier sont justes) : "
                                                    "vérifie chaque point rendu, avec plusieurs pas"),
    (middle_points_off_c24, "affine_is_exact_24", "aux points intermédiaires (le premier et le dernier sont justes) : "
                                                  "vérifie chaque point rendu, avec plusieurs pas"),
    (one_step_only_c24, "constant_is_exact_24", "dès deux pas (un seul pas est juste) : essaie plusieurs nombres de pas"),
    (one_step_only_c24, "affine_is_exact_24", "dès deux pas (un seul pas est juste) : essaie plusieurs nombres de pas"),
    (unit_speed_c24, "constant_is_exact_24", "sauf pour les vitesses 0 et 1 : essaie plusieurs vitesses, dont des "
                                             "négatives et des non entières"),
    (unit_slope_c24, "affine_is_exact_24", "sauf pour une pente κ₁ = 1 : essaie plusieurs champs κ₀ + κ₁ t"),
    (slightly_off_c24, "constant_is_exact_24", "de 5·10⁻⁹ + 5·10⁻⁶ |x| : la tolérance de l'énoncé est 10⁻⁹ (np.allclose a "
                                               "par défaut rtol = 1e-5 et atol = 1e-8)"),
    (slightly_off_c24, "affine_is_exact_24", "de 5·10⁻⁹ + 5·10⁻⁶ |x| : la tolérance de l'énoncé est 10⁻⁹ (np.allclose a "
                                             "par défaut rtol = 1e-5 et atol = 1e-8)"),
    (frozen_time_c24, "error_halves_24", "pour un champ qui dépend du temps (elle lit le champ en t_0, et son erreur ne "
                                         "tend pas vers 0) : ton champ dépend-il de t, et ton test exige-t-il un rapport "
                                         "entre 1,8 et 2,2 ?"),
    (frozen_point_c24, "error_halves_24", "pour un champ qui dépend de x (elle lit le champ au point de départ, et son "
                                          "erreur ne tend pas vers 0) : ton champ dépend-il de x, et ton test exige-t-il "
                                          "un rapport entre 1,8 et 2,2 ?"),
    (halfway_c24, "error_halves_24", "partout : son erreur au temps final ne tend pas vers 0 ; le rapport des erreurs "
                                     "doit être entre 1,8 et 2,2, pas seulement plus grand que 1"),
]

with checked("MB5.24"):
    results_24 = []
    report_24 = reporter(results_24, stop=False)
    TESTS_C24 = {"constant_is_exact_24": constant_is_exact_24, "affine_is_exact_24": affine_is_exact_24,
                 "error_halves_24": error_halves_24}

    def anonymous_c24(method):
        """method under a neutral name: a test must judge what the method does, not what it is called."""
        def method_c24(f, x0, t0, t1, n_steps):
            return method(f, x0, t0, t1, n_steps)
        method_c24.__name__ = method_c24.__qualname__ = "method"
        return method_c24

    def verdict_c24(name, label, method):
        got = call_safely(f"Ex MB5.24 : {name}({label})", TESTS_C24[name], anonymous_c24(method))
        if not isinstance(got, (bool, np.bool_)):
            if isinstance(got, (int, np.integer)):
                what = f"{int(got)} (un entier)"
            elif isinstance(got, (float, np.floating)):
                what = f"{float(got):g} (un nombre)"
            else:
                what = shown(got)
            print(f"❌ Ex MB5.24 : {name}({label}) rend {what}, pas True ou False")
            raise StopCheck
        return bool(got)

    def verdicts_c24(label, method):
        return [verdict_c24(name, label, method) for name in TESTS_C24]

    on_ref_c24 = [verdicts_c24("euler_ode de la référence", ref_gp.euler_ode),
                  verdicts_c24("midpoint_ode de la référence", ref_gp.midpoint_ode)]
    report_24(passes("MB5.24a", on_ref_c24), "tes trois tests, sur euler_ode et midpoint_ode de la référence",
              "tes trois tests ne rendent pas, sur euler_ode et midpoint_ode de la référence, ce que dit la fiche §8 de "
              "ces deux méthodes : relis les propriétés de chacune (et, pour error_halves_24, prends N assez grand)")
    twins_c24 = [verdicts_c24("une autre écriture juste d'Euler", euler_twin_c24),
                 verdicts_c24("une autre écriture juste du point milieu", midpoint_twin_c24)]
    report_24(twins_c24 == on_ref_c24, "deux autres écritures justes des deux méthodes reçoivent les mêmes verdicts",
              "deux autres écritures justes des deux méthodes (les instants accumulés pas à pas : des arrondis de l'ordre de "
              "10⁻¹⁶) ne reçoivent pas les mêmes verdicts que euler_ode et midpoint_ode : ta tolérance est-elle trop "
              "stricte ?")
    for number_c24, (method_c24, name_c24, clause_c24) in enumerate(WRONG_C24, start=1):
        refused_c24 = not verdict_c24(name_c24, f"méthode fausse cachée n° {number_c24}", method_c24)
        report_24(refused_c24, f"{name_c24} refuse la méthode fausse cachée n° {number_c24}",
                  f"{name_c24} accepte la méthode fausse cachée n° {number_c24}, fausse {clause_c24}")
    verdict("MB5.24", all(results_24) and len(results_24) == 2 + len(WRONG_C24), "tes trois tests jugent juste les deux "
            "méthodes de la référence, deux autres écritures justes, et refusent les méthodes fausses cachées.",
            "relis les lignes ❌ ci-dessus.")

**Ce qu'il faut regarder** : pourquoi `affine_is_exact_24` refuse-t-il la méthode d'Euler, et `error_halves_24` le point milieu ? Pourquoi tester chaque point rendu, et pas seulement le dernier ? Les réponses sont dans `05_solutions.md`, section Notebook.

## Partie D · Défi

Fiche §8 et §9. Transporter un bruit vers une loi à deux bosses par son champ moyen exact, avec huit évaluations du champ, vérifié sur des lois cachées tirées à chaque exécution.

### Ex MB5.25 — Défi : transporter un bruit N(0, 1) vers une loi à deux bosses par son champ moyen exact, en au plus huit évaluations du champ, sous une distance de Wasserstein fixée 🏆 ★★★ ⏱️ 35 min
**Objectif :** transporter un bruit vers une loi à deux bosses par son champ moyen exact, avec un budget d'évaluations du champ, et le vérifier sur des lois cachées.  
**Prérequis :** fiche §8 et §9 ; M27 (distance de Wasserstein) ; Ex MB5.17, MB5.21 · **Parcours :** C

Le notebook fournit `MIX_25`, une loi à deux bosses (les poids, les moyennes et les variances d'un mélange de deux gaussiennes de dimension 1), `Z_25`, 20 000 tirages de $\mathcal{N}(0, 1)$, et `TARGET_25`, 200 000 tirages de `MIX_25`.
- a) Écris `field_25(x, s, weights, means, variances)`, le champ moyen exact $v(x, s)$ du chemin droit vers ce mélange, avec `posterior_mean_mixture` (pour $0 < s \le 1$ ; `x` est un tableau de points).
- Puis écris `transport_25(field, z)` : depuis les points `z` (en $s = 1$), suis $x' = v(x, s)$ jusqu'en $s = 0$, en appelant `field(x, s)` **au plus huit fois** (un appel sur un tableau entier compte pour un ; `field` prend le point d'abord, puis l'instant), et rends les points atteints. Ta fonction ne connaît le mélange que par `field` ; tout NumPy et SciPy sont permis.

**Critère de réussite :** avec au plus huit appels du champ pour chaque loi, la distance de Wasserstein $W_1$ (M27) entre tes points et la loi à deux bosses, divisée par l'écart-type de cette loi, est au plus $0{,}07$ pour `MIX_25`, et au plus $0{,}07$ en moyenne pour trois séries de cinq lois cachées, tirées à nouveau à chaque exécution (la moyenne des quinze rapports) ; chaque série a une loi de chacun de ces cinq types : facile ; deux bosses très étroites ; une bosse légère d'une autre largeur ; deux bosses plus écartées que le bruit n'est large ; deux bosses qui se chevauchent. La vérification passe à `transport_25` un `field` qu'elle calcule elle-même, et compte ses appels. Ta fonction ne doit pas exécuter le code du module caché du défi (`solutions/challenges/mb5_25.py`) : la vérification le contrôle. Relève :
- b) avec `scipy.stats.wasserstein_distance` entre tes points et `TARGET_25`, la distance obtenue par un seul pas d'Euler (`euler_ode`, de $s = 1$ à $s = 0$, avec ton `field_25` ; `euler_ode` appelle `f(s, x)` : passe-lui `lambda s, x: field_25(x, s, *MIX_25)`), puis par huit pas d'Euler, puis par ta fonction (une liste de 3 nombres) ;
- c) le nombre d'appels de `field` que fait ta fonction (un entier).

Point de départ : huit pas d'Euler ne suffisent pas ; la fiche §8 dit comment comparer deux méthodes qui ont droit au même nombre d'évaluations.

In [ ]:
# The two-bump law of the challenge: the weights, the means and the variances of a mixture of two Gaussians
MIX_25 = ((0.38, 0.62), (-1.6, 2.1), (0.14, 0.3))
Z_25 = frozen(np.random.default_rng(7725).normal(size=20_000))      # 20 000 draws of N(0, 1): the points at s = 1
_rng_25 = np.random.default_rng(7726)                                # 200 000 draws of MIX_25
_bump_25 = _rng_25.choice(2, size=200_000, p=MIX_25[0])
TARGET_25 = frozen(np.array(MIX_25[1])[_bump_25] + np.sqrt(np.array(MIX_25[2]))[_bump_25] * _rng_25.normal(size=200_000))
del _rng_25, _bump_25

In [ ]:
def field_25(x, s, weights, means, variances):
    """The exact mean field v(x, s) of the straight path to the mixture (weights, means, variances), at the points x (an
    array), for 0 < s <= 1."""
    raise NotImplementedError("field_25 n'est pas encore écrite (04_indices.md si tu bloques)")


def transport_25(field, z):
    """From the points z (at s = 1), follow x' = v(x, s) down to s = 0, calling field(x, s) at most 8 times (one call for
    a whole array counts once); returns the points reached."""
    raise NotImplementedError("transport_25 n'est pas encore écrite (04_indices.md si tu bloques)")


answer_MB5_25b = answer_MB5_25c = ...
with wbm.attempt("MB5.25"):          # your functions: ⏳ while they are not written
    answer_MB5_25b = ...  # b) a list of 3 numbers, as computed: wasserstein_distance to TARGET_25 of one Euler step, of eight Euler steps, of transport_25
    answer_MB5_25c = ...  # c) an integer: the number of calls of field made by transport_25

In [ ]:
FIELD_CASES_C25 = {'cases': [[[-2.13, -0.41, 0.76, 2.58], 1.0, ((0.38, 0.62), (-1.6, 2.1), (0.14, 0.3))],
           [[-1.87, 0.33, 1.97], 0.63, ((0.38, 0.62), (-1.6, 2.1), (0.14, 0.3))],
           [[-1.52, 0.18, 2.27], 0.21, ((0.38, 0.62), (-1.6, 2.1), (0.14, 0.3))],
           [[-1.71, 1.98, 2.36], 0.047, ((0.38, 0.62), (-1.6, 2.1), (0.14, 0.3))],
           [[-0.94, 0.61, 1.83], 0.38, ((0.26, 0.74), (-0.7, 1.45), (0.09, 0.4))],
           [[-2.6, -0.15, 1.2, 3.05], 0.82, ((0.3, 0.45, 0.25), (-1.9, 0.35, 2.4), (0.2, 0.05, 0.33))]],
 'id': 'MB5.25e',
 'id_b': 'MB5.25d'}   # hidden points, times and mixtures for field_25, and the sub-ID expected


def check_c25():
    """The check of MB5.25, in a function. The hidden laws live in another process (the hidden module run as a script):
    it answers each call of the field, counts the calls, and measures the points reached only once transport_25 has
    seen all the laws; neither a hidden law nor what makes it (a seed, a kind) lives in this process while the learner's
    function runs. Each call runs under code_from: the learner's function may not run code of the hidden module."""
    import json
    import subprocess
    import sys
    from scipy.stats import wasserstein_distance
    hidden_file = wbm.repo_root() / "solutions" / "challenges" / "mb5_25.py"   # reading it spoils the challenge
    twins = code_keys_of_file(hidden_file)                      # its functions, to recognise a copy run from elsewhere
    results = []
    report = reporter(results, stop=False)

    # a) field_25 at hidden points (other times, other mixtures)
    got = []
    for case_points, case_s, case_parts in FIELD_CASES_C25["cases"]:
        x = np.array(case_points)                               # non-dyadic floats: a change in place would show
        out = as_floats(call_safely("Ex MB5.25 : field_25 (appel caché)", field_25, x, case_s, *case_parts))
        if not np.array_equal(x, np.array(case_points)):
            print("❌ Ex MB5.25 : field_25 modifie le tableau x reçu : calcule sur une copie")
            raise StopCheck
        if out is None or out.shape != x.shape:
            print(f"❌ Ex MB5.25 : field_25 rend un tableau de la forme de x, reçu {shown(out)}")
            raise StopCheck
        got += out.tolist()
    field_ok = passes(FIELD_CASES_C25["id"], got)
    report(field_ok, "Ex MB5.25a : field_25 en des points cachés (d'autres instants, d'autres mélanges)",
           "Ex MB5.25a : field_25 en des points cachés : ce n'est pas le champ moyen v(x, s) = (x − E[x_0 | x_s = x])/s, "
           "avec le débruiteur posterior_mean_mixture de gains a = 1 − s et b = s (fiche §9) ; si c'est bien ce que tu "
           "calcules, les tests de ta posterior_mean_mixture passent-ils tous ?")

    def exactly(stream, size):
        data = stream.read(size)
        if len(data) != size:
            raise OSError("échange interrompu")
        return data

    def transport(label, field, z):
        """transport_25(field, z) under code_from: the points reached, as a float array of the shape of z."""
        with code_from([hidden_file], twins=twins, escapes=True) as seen:
            out = call_safely(label, transport_25, field, z.copy())
        if seen and seen[0].startswith("lance "):
            print(f"❌ {label} : transport_25 {seen[0].replace(' : ', ' (', 1)}) : le transport se calcule dans ta "
                  "fonction ; ce qu'exécute un autre processus échappe à la vérification")
            raise StopCheck
        if seen:
            what = "lit" if seen[0].startswith("ouvre ") else "exécute du code de"
            where = "" if seen[0].startswith("ouvre ") else f" ({seen[0]})"
            print(f"❌ {label} : transport_25 {what} solutions/challenges/mb5_25.py{where}, la partie cachée du défi, ou "
                  "une copie de son code : la vérification mesurerait sa méthode, pas la tienne")
            raise StopCheck
        points = as_floats(out)
        if points is None or points.shape != z.shape:
            print(f"❌ {label} : transport_25 rend un tableau de {z.size} points, un par point de z, reçu {shown(out)}")
            raise StopCheck
        return points

    server = subprocess.Popen([sys.executable, "-I", str(hidden_file), "--serve"], stdin=subprocess.PIPE,
                              stdout=subprocess.PIPE, stderr=subprocess.DEVNULL)   # this kernel's Python, here or on Colab
    try:
        count = json.loads(exactly(server.stdout, 1) + server.stdout.readline())["count"]

        def field_of(k):
            """The field v(x, s) of the law k, computed by the other process (each call is counted there)."""
            def field(x, s):
                if isinstance(s, (bool, np.bool_)) or np.ndim(s) != 0 or not 0 < float(s) <= 1:
                    raise ValueError(f"le champ prend un instant s de ]0, 1] (un nombre), reçu {shown(s)}")
                x = np.asarray(x, dtype=float)
                flat = np.ascontiguousarray(x.ravel(), dtype="<f8")
                server.stdin.write(f"F {k} {float(s)!r} {flat.size}\n".encode() + flat.tobytes())
                server.stdin.flush()
                return np.frombuffer(exactly(server.stdout, 8 * flat.size), dtype="<f8").reshape(x.shape).copy()
            return field

        def send(k, points):
            flat = np.ascontiguousarray(points.ravel(), dtype="<f8")
            server.stdin.write(f"R {k} {flat.size}\n".encode() + flat.tobytes())
            server.stdin.flush()

        mine = transport("Ex MB5.25 : transport_25 (le mélange du notebook)", field_of(count - 1), np.array(Z_25))
        send(count - 1, mine)
        rng = np.random.default_rng()                           # new noise points at each run
        for k in range(count - 1):
            send(k, transport(f"Ex MB5.25 : transport_25 (loi cachée {k + 1})", field_of(k), rng.normal(size=20_000)))
        server.stdin.write(b"verdict\n")                       # only now: the measures, the kinds and the counts
        server.stdin.flush()
        laws = json.loads(exactly(server.stdout, 1) + server.stdout.readline())
        assert len(laws) == count
    except (OSError, ValueError, KeyError, TypeError, AssertionError) as exc:
        print("❌ Ex MB5.25 : la vérification n'a pas pu échanger les points et les champs avec le processus qui tire les "
              f"lois cachées ({type(exc).__name__} ; ta fonction a-t-elle touché à ce processus ?) : exécute à nouveau la "
              "cellule")
        raise StopCheck from None
    finally:                                                    # always, even on an error or an interruption
        if server.poll() is None:
            server.kill()
        server.wait()
        for pipe in (server.stdin, server.stdout):
            try:
                pipe.close()
            except OSError:                                     # a pipe already broken
                pass

    advice = (" : avec ce budget, ta méthode ne suit pas les trajectoires d'assez près ; la fiche §8 dit quelle méthode "
              "est la plus précise à nombre d'évaluations égal")
    notebook, hidden = laws[-1], laws[:-1]
    if notebook["rel"] is None:
        report(False, "", "le mélange du notebook : transport_25 rend des points qui ne sont pas tous finis")
    else:
        report(notebook["rel"] <= 0.07 and notebook["calls"] <= 8,
               f"le mélange du notebook : W₁ = {fr(notebook['rel'], 3)} fois l'écart-type, en {notebook['calls']} appels "
               "du champ", "le mélange du notebook : " + " et ".join(
                   ([f"{notebook['calls']} appels du champ, plus que 8"] if notebook["calls"] > 8 else [])
                   + ([f"W₁ = {fr(notebook['rel'], 3)} fois l'écart-type, au-dessus de 0,07" + advice]
                      if notebook["rel"] > 0.07 else [])))
    for number in sorted({law["series"] for law in hidden}):        # what each hidden law measured (no verdict)
        series = [law for law in hidden if law["series"] == number]
        print(f"   série {number} : " + " ; ".join(
            f"« {law['kind']} » {'non finis' if law['rel'] is None else fr(law['rel'], 3)} ({law['calls']} appels)"
            for law in series))
    over = [law for law in hidden if law["calls"] > 8]
    broken = [law for law in hidden if law["rel"] is None]
    report(not over and not broken, f"les {len(hidden)} lois cachées : huit appels du champ au plus pour chacune, des "
           "points finis", f"les {len(hidden)} lois cachées : " + " ; ".join(
               ([f"{len(over)} lois avec plus de 8 appels du champ (jusqu'à {max(law['calls'] for law in over)})"]
                if over else []) + ([f"{len(broken)} lois avec des points qui ne sont pas tous finis"] if broken else [])))
    mean = float(np.mean([law["rel"] for law in hidden])) if not broken else math.inf
    calls_nb = notebook["calls"]
    if calls_nb == 0:
        tuned = " ; ta fonction n'appelle jamais field : elle ne connaît du mélange que ce que le notebook lui montre"
    elif notebook["rel"] is not None and notebook["rel"] <= 0.07 and notebook["rel"] < 0.25 * mean:
        tuned = (" ; elle réussit pourtant le mélange du notebook, bien mieux que les lois cachées : se règle-t-elle sur "
                 "MIX_25 au lieu de suivre field ?")
    elif calls_nb < 8:
        tuned = f" ; elle n'emploie que {calls_nb} des 8 appels permis : des pas plus courts réduiraient l'erreur"
    else:
        tuned = ""
    report(mean <= 0.07, f"les {len(hidden)} lois cachées : W₁ moyen = {fr(mean, 3)} fois l'écart-type, au plus 0,07",
           f"les {len(hidden)} lois cachées : W₁ moyen = {fr(mean, 3)} fois l'écart-type, au-dessus de 0,07" + advice
           + tuned if math.isfinite(mean) else f"les {len(hidden)} lois cachées : W₁ ne se mesure pas sur des points non finis")
    criterion = all(results) and len(results) == 4               # field_25, the notebook, the budget, the mean

    # b) and c): the two Euler distances (hidden sub-ID), the third and the count recomputed with your function
    if not filled(answer_MB5_25b):
        print("⏳ Ex MB5.25b : pas encore fait.")
    else:
        mine_b = as_floats(answer_MB5_25b)
        if mine_b is None or mine_b.shape != (3,):
            report(False, "", f"Ex MB5.25b : une liste de 3 nombres, reçu {shown(answer_MB5_25b)}")
        else:
            report(passes(FIELD_CASES_C25["id_b"], mine_b[:2].tolist()),
                   "Ex MB5.25b : les distances d'un pas et de huit pas d'Euler",
                   "Ex MB5.25b : les deux premières distances ne sont pas celles d'un pas, puis de huit pas égaux de "
                   "euler_ode, de s = 1 à s = 0, depuis Z_25, avec ton field_25 pour MIX_25"
                   + ("" if field_ok else " (corrige d'abord field_25)"))
            again = float(wasserstein_distance(mine, TARGET_25))
            report(math.isclose(float(mine_b[2]), again, rel_tol=1e-6, abs_tol=1e-9),
                   "Ex MB5.25b : la troisième distance est celle des points de ta fonction",
                   "Ex MB5.25b : la troisième distance n'est pas celle des points que rend transport_25 maintenant, avec "
                   "le champ exact (la vérification la recalcule) : exécute à nouveau ta cellule"
                   + ("" if field_ok else " (et corrige d'abord field_25)"))
    if not filled(answer_MB5_25c):
        print("⏳ Ex MB5.25c : pas encore fait.")
    else:
        report(isinstance(answer_MB5_25c, (int, np.integer)) and not isinstance(answer_MB5_25c, (bool, np.bool_))
               and int(answer_MB5_25c) == notebook["calls"], "Ex MB5.25c : le nombre d'appels de ta fonction",
               "Ex MB5.25c : ta réponse n'est pas le nombre d'appels de field que fait transport_25 sur le mélange du "
               "notebook (un entier ; la vérification les compte)")
    verdict("MB5.25", criterion, "critère atteint : en huit appels du champ au plus, W₁ au plus 0,07 fois l'écart-type "
            "pour le mélange du notebook, et en moyenne pour les quinze lois cachées.",
            "critère pas encore atteint : relis les lignes ❌ ci-dessus.")


with checked("MB5.25"):
    check_c25()

**Pour réfléchir** (une fois le critère atteint) : pourquoi huit pas d'Euler ne suffisent-ils pas, alors que ta méthode, pour le même nombre d'appels, suffit ? Pourquoi une méthode réglée sur `MIX_25` seul (ses quantiles, par exemple) ne suffit-elle pas pour les lois cachées ? Les réponses sont dans `05_solutions.md`, section Notebook.

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu propager la moyenne et la variance d'une récurrence linéaire gaussienne, trouver sa loi stationnaire, et dire combien de pas il faut pour oublier le départ ?
2. Sais-tu conditionner une marche aléatoire sur deux instants, et construire des trajectoires de mêmes lois marginales mais de corrélations différentes ?
3. Sais-tu passer du SNR à la part de variance, au logit et aux décibels ?
4. Sais-tu dire ce qu'apprend la MSE, calculer le débruiteur optimal d'un mélange, et le relier au score par la formule de Tweedie ?
5. Sais-tu écrire l'ELBO d'une chaîne de latents, et calculer une loi guidée et sa condition d'existence ?
6. Sais-tu suivre une équation différentielle par Euler et par le point milieu, comparer leurs erreurs à nombre d'évaluations égal, et transporter une loi par le champ moyen d'un chemin droit ?

**La suite** : le chapitre bonus B5 de workbookIA, qui applique ces outils.